In [ ]:
# Cell 176 (08_story_extras.ipynb, cell 1): FRESH KERNEL. Phase 7b guards + input inspection. READ-ONLY, writes NOTHING.
def _c176():
    import os, sys, re, json, pickle, hashlib, warnings
    import numpy as np, pandas as pd
    warnings.filterwarnings("ignore")
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
    for rel, h in {"shared/src/masking.py": "c6094c2b232e", "shared/src/rq3.py": "9ccbba66073b", "shared/src/safe_save.py": "afff88e2b30a", "shared/src/checks.py": "ef542907c037"}.items():
        assert sha(R + "/" + rel).startswith(h), "hash mismatch for %s: STOP" % rel
    print("guards OK: PYTHONHASHSEED=0 and four module hashes")
    L = lambda p: pickle.load(open(p, "rb")); FV = R + "/full_variable"
    imp = L(FV + "/models/imputer.pkl")
    print("\nlocked imputer: n_features_in_ %s | feature_names_in_ %s" % (imp.n_features_in_, list(getattr(imp, "feature_names_in_", []))))
    print("  'Default' among imputer input columns:", "Default" in list(getattr(imp, "feature_names_in_", [])))
    nb = json.load(open(FV + "/notebooks/Expirements_FROZEN.ipynb"))
    for ci, c in enumerate(nb["cells"]):
        if c.get("cell_type") != "code": continue
        s = "".join(c["source"]) if isinstance(c["source"], list) else c["source"]
        if "IterativeImputer" in s and ".fit" in s:
            print("\n--- frozen cell %d (imputer fit) ---" % ci); print(s[:2200])
    lr = L(FV + "/models/lr_final.pkl"); rf = L(FV + "/models/rf_final_restricted.pkl")
    print("\nlocked LR params:", {k: lr.get_params()[k] for k in ("C", "penalty", "solver", "class_weight", "max_iter", "tol", "random_state")}, "| n_iter_", lr.n_iter_)
    print("locked RF params:", {k: rf.get_params()[k] for k in ("n_estimators", "max_depth", "min_samples_leaf", "class_weight", "random_state", "n_jobs", "max_features")})
    sb = L(R + "/selection/predictions/sel_bn_test_predictions.pkl"); bb = L(FV + "/predictions/bn_test_predictions.pkl"); ab = L(FV + "/predictions/ablation_predictions.pkl")
    print("\nsel_bn_test_predictions keys:", list(sb.keys()), "| index sample", list(sb["index"])[:4], type(sb["index"][0]).__name__)
    p = bb["bn_test_probs"]; print("bn_test_predictions: bn_test_probs", type(p).__name__, len(p), "| index sample", list(p.index[:4]), "| NaN count", int(p.isna().sum()), "| NaN labels", list(p.index[p.isna()]))
    print("ablation keys:", list(ab.keys()))
    sp = L("%s/Data/processed/split_indices.pkl" % R); print("split keys:", list(sp.keys()), "| test_idx sample", list(sp["test_idx"])[:4])
    ix_s = pd.Index(sb["index"]); ix_m = p.index
    print("selection and monitoring test index identical (labels, order):", bool(ix_s.equals(ix_m)), "| both equal split test_idx:", bool(ix_s.equals(pd.Index(sp["test_idx"]))))
    valid = p.notna().to_numpy(); print("BN-defined rows (monitoring):", int(valid.sum()), "| selection included all:", bool(np.all(sb["included"])))
    print("\nselection pickles present:", {f: os.path.exists(R + "/selection/predictions/" + f) for f in ("sel_baseline_test_predictions.pkl", "sel_bn_test_predictions.pkl")})
    bp = L(R + "/selection/predictions/sel_baseline_test_predictions.pkl"); print("sel_baseline keys:", list(bp.keys()), "| tau", bp["tau"])
    for f in ("X_train_df.pkl", "X_test_df.pkl", "df_train.pkl", "df_test.pkl", "df_train_lr_rf.pkl"):
        print("  Data/processed/%s: %s" % (f, "present" if os.path.exists(R + "/Data/processed/" + f) else "MISSING"))
    print("\nfull-design (R4) check: X_train_df cols", L(R + "/Data/processed/X_train_df.pkl").shape[1], "| sel_additive_lbfgs_diagnostic:", pd.read_csv(R + "/selection/tables/sel_additive_lbfgs_diagnostic.csv")[["auc"]].round(4).values.tolist())
_c176()

In [ ]:
# Cell 177 (08_story_extras.ipynb): shared/src/decision_value.py + validation against existing numbers. Saves the module only if all checks pass.
DV_SRC = '''"""Decision-value helpers (descriptive). cost(y, flag, r) = r*FN + FP. flag = p >= tau.
Trivial policies: accept_all (flag nobody) = r*D; reject_all (flag everyone) = N - D. Primary baseline = the cheaper of the two."""
import numpy as np
import pandas as pd

GRID = np.round(np.arange(1, 100) / 100.0, 2)


def cost(y, flag, r):
    y = np.asarray(y).astype(int); flag = np.asarray(flag).astype(bool)
    return int(r * ((~flag) & (y == 1)).sum() + (flag & (y == 0)).sum())


def confusion(y, flag):
    y = np.asarray(y).astype(int); flag = np.asarray(flag).astype(bool)
    return int((flag & (y == 1)).sum()), int((flag & (y == 0)).sum()), int((~flag & (y == 1)).sum()), int((~flag & (y == 0)).sum())


def trivial(y, r):
    y = np.asarray(y).astype(int); D = int(y.sum()); N = len(y)
    a, b = r * D, N - D
    return dict(accept_all=int(a), reject_all=int(b), cheaper=int(min(a, b)), cheaper_policy="accept_all" if a <= b else "reject_all")


def saving(cost_model, cheaper):
    return 1.0 - cost_model / cheaper


def oof_tau(oof, y_train, r, tie="highest"):
    """Cost-minimising tau on the 0.01 grid from OUT-OF-FOLD train predictions. tie: 'highest' (spec) or 'lowest'."""
    c = np.array([cost(y_train, np.asarray(oof) >= t, r) for t in GRID])
    best = GRID[c == c.min()]
    return float(best.max() if tie == "highest" else best.min()), int(c.min()), (float(best.min()), float(best.max()))


def curve(y, preds, taus_by_model, rs):
    """preds: {model: p}; taus_by_model: {model: {r: tau}}. Returns one row per (model, r)."""
    rows = []
    for r in rs:
        tr = trivial(y, r)
        for m, p in preds.items():
            t = taus_by_model[m][r]; fl = np.asarray(p) >= t; c = cost(y, fl, r); TP, FP, FN, TN = confusion(y, fl)
            rows.append(dict(r=r, model=m, tau=t, flagged=int(fl.sum()), TP=TP, FP=FP, FN=FN, TN=TN, cost=c, accept_all=tr["accept_all"], reject_all=tr["reject_all"],
                             cheaper_policy=tr["cheaper_policy"], cost_cheaper_trivial=tr["cheaper"], saving=saving(c, tr["cheaper"])))
    return pd.DataFrame(rows)
'''

def _c177():
    import os, sys, pickle, hashlib, importlib.util, warnings
    import numpy as np, pandas as pd
    warnings.filterwarnings("ignore")
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sys.path.insert(0, R + "/shared/src"); import safe_save as ss
    mp = R + "/shared/src/decision_value.py"; assert not os.path.exists(mp), "decision_value.py exists: STOP"
    ns = {}; exec(compile(DV_SRC, "dv_candidate", "exec"), ns)
    L = lambda p: pickle.load(open(p, "rb"))
    sp = L(R + "/Data/processed/split_indices.pkl"); y_tr = np.asarray(sp["y_train"]).astype(int)
    sb = L(R + "/selection/predictions/sel_bn_test_predictions.pkl"); bp = L(R + "/selection/predictions/sel_baseline_test_predictions.pkl")
    y = np.asarray(sb["y"]).astype(int); pbn = np.asarray(sb["p"], float)
    assert np.array_equal(y, np.asarray(bp["y"]).astype(int)) and np.array_equal(pbn, np.asarray(bp["p_bn"], float))
    ok = True
    def chk(name, got, exp):
        nonlocal ok; good = (got == exp); ok &= bool(good); print("  %-60s got %-22s expected %-22s %s" % (name, got, exp, "PASS" if good else "FAIL"))
    print("selection BN test cost at tau = 1/(1+r):")
    for r, e in zip((5, 10, 15, 20), (10025, 18069, 21788, 23602)):
        chk("BN cost r=%d" % r, ns["cost"](y, pbn >= 1.0 / (1 + r), r), e)
    print("selection OOF taus (saved Phase 5 OOF predictions, same params):")
    for m, key in (("LR", "oof_lr"), ("RF", "oof_rf")):
        for r, e in ((5, {"LR": 0.90, "RF": 0.69}[m]), (10, 0.57)):
            lo = ns["oof_tau"](bp[key], y_tr, r, "lowest"); hi = ns["oof_tau"](bp[key], y_tr, r, "highest")
            chk("%s OOF tau r=%d (lowest-tie rule, as used in Phases 5/6b)" % (m, r), round(lo[0], 2), e)
            print("      plateau %.2f..%.2f | highest-tie rule gives %.2f" % (lo[2][0], lo[2][1], hi[0]))
            ft = lambda t: int((np.asarray(bp["p_" + m.lower()]) >= t).sum())
            if abs(hi[0] - lo[0]) > 1e-9: print("      test rows flagged at %.2f: %d | at %.2f: %d | same cost on test: %s" % (lo[0], ft(lo[0]), hi[0], ft(hi[0]), ns["cost"](y, np.asarray(bp["p_" + m.lower()]) >= lo[0], r) == ns["cost"](y, np.asarray(bp["p_" + m.lower()]) >= hi[0], r)))
    chk("selection RF test cost r=5 at tau 0.69", ns["cost"](y, np.asarray(bp["p_rf"]) >= 0.69, 5), 9918)
    ab = L(R + "/full_variable/predictions/ablation_predictions.pkl"); yt = np.asarray(sp["y_test"]).astype(int)
    fl = np.asarray(ab["LR_all"], float) >= 0.69
    chk("monitoring locked LR cost at tau 0.69, 30,000 rows", ns["cost"](yt, fl, 5), 6473)
    chk("monitoring locked LR confusion TP/FP/FN/TN", ns["confusion"](yt, fl), (1139, 2143, 866, 25852))
    t = ns["trivial"](y, 10); print("  trivial policies selection r=10:", t)
    print("\nVALIDATION:", "ALL PASS" if ok else "MISMATCH: STOP, module not saved")
    if not ok: return
    with open(mp, "x", encoding="utf-8") as f: f.write(DV_SRC)
    h = hashlib.sha256(open(mp, "rb").read()).hexdigest(); assert h == hashlib.sha256(DV_SRC.encode()).hexdigest(); os.chmod(mp, 0o444)
    spec = importlib.util.spec_from_file_location("dv", mp); m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
    assert m.cost(y, pbn >= 1 / 6, 5) == 10025
    print("saved shared/src/decision_value.py sha12 %s (re-imported from disk, passes)" % h[:12])
_c177()

In [ ]:
# Cell 178 (08_story_extras.ipynb): Part B decision-value curves (both contributions). Writes 4 files + mon_oof_predictions.pkl (exclusive create). No restart.
def _c178():
    import os, sys, io, time, pickle, hashlib, importlib.util, textwrap, warnings
    import numpy as np, pandas as pd
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    from sklearn.base import clone
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score
    warnings.filterwarnings("ignore")
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sys.path.insert(0, R + "/shared/src"); import safe_save as ss
    def load(name, rel):
        s = importlib.util.spec_from_file_location(name, R + "/" + rel); m = importlib.util.module_from_spec(s); s.loader.exec_module(m); return m
    dv = load("dv", "shared/src/decision_value.py"); ck = load("checks", "shared/src/checks.py")
    assert hashlib.sha256(open(R + "/shared/src/decision_value.py", "rb").read()).hexdigest().startswith("c1476cef61d1")
    OUT = {"sel": "selection/tables/sel_decision_value_selection.csv", "mon": "monitoring_rq3/mon_decision_value_monitoring.csv",
           "comb": "selection/tables/sel_decision_value_combined.csv", "fig": "selection/figures/sel_decision_value_combined.png"}
    for rel in OUT.values(): ss.check_path(os.path.join(R, rel))
    OOFP = R + "/monitoring_rq3/mon_oof_predictions.pkl"
    L = lambda p: pickle.load(open(p, "rb")); RS = list(range(1, 21)); SP = (5, 10, 15, 20)
    sp = L(R + "/Data/processed/split_indices.pkl"); y_tr = np.asarray(sp["y_train"]).astype(int); y_te = np.asarray(sp["y_test"]).astype(int)

    # ---- selection
    sb = L(R + "/selection/predictions/sel_bn_test_predictions.pkl"); bp = L(R + "/selection/predictions/sel_baseline_test_predictions.pkl")
    ys = np.asarray(sb["y"]).astype(int); assert np.array_equal(ys, y_te)
    Ps = {"BN": np.asarray(sb["p"], float), "LR": np.asarray(bp["p_lr"], float), "RF": np.asarray(bp["p_rf"], float)}
    Ts = {"BN": {r: 1.0 / (1 + r) for r in RS}, "LR": {r: dv.oof_tau(bp["oof_lr"], y_tr, r, "highest")[0] for r in RS}, "RF": {r: dv.oof_tau(bp["oof_rf"], y_tr, r, "highest")[0] for r in RS}}
    sel = dv.curve(ys, Ps, Ts, RS); sel.insert(0, "contribution", "selection"); sel["n_rows"] = len(ys)

    # ---- monitoring: confirm imputer feature-only, then OOF train predictions
    imp = L(R + "/full_variable/models/imputer.pkl"); assert "Default" not in list(imp.feature_names_in_), "target in imputer: STOP"
    lr0 = L(R + "/full_variable/models/lr_final.pkl"); rf0 = L(R + "/full_variable/models/rf_final_restricted.pkl")
    Xtr = L(R + "/Data/processed/X_train_df.pkl").values
    if os.path.exists(OOFP):
        oof = L(OOFP); print("loaded saved monitoring OOF predictions")
    else:
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42); oof = {"LR": np.zeros(len(y_tr)), "RF": np.zeros(len(y_tr))}; t0 = time.time()
        lrp = {k: lr0.get_params()[k] for k in ("C", "penalty", "class_weight")}
        for k, (a, b) in enumerate(cv.split(Xtr, y_tr)):
            m = ck.fit_converged(LogisticRegression(solver="lbfgs", max_iter=5000, **lrp), Xtr[a], y_tr[a], name="mon LR fold %d" % k); oof["LR"][b] = m.predict_proba(Xtr[b])[:, 1]
            m = clone(rf0).set_params(n_jobs=1).fit(Xtr[a], y_tr[a]); oof["RF"][b] = m.predict_proba(Xtr[b])[:, 1]
            print("  monitoring OOF fold %d done (%.0fs)" % (k, time.time() - t0))
        pk = ss.check_path(OOFP); blob = pickle.dumps(dict(oof=oof, folds="5-fold stratified, shuffle, seed 42", lr="lbfgs C=100 l2 balanced max_iter 5000 (fit_converged)", rf="locked 500/15/10 n_jobs=1",
                                                         note="locked imputer fitted on all train rows, so OOF is not fully out-of-sample w.r.t. imputation"))
        with open(pk, "xb") as f: f.write(blob)
    print("monitoring OOF AUC: LR %.4f | RF %.4f" % (roc_auc_score(y_tr, oof["LR"]), roc_auc_score(y_tr, oof["RF"])))
    bb = L(R + "/full_variable/predictions/bn_test_predictions.pkl")["bn_test_probs"]; ab = L(R + "/full_variable/predictions/ablation_predictions.pkl")
    v = bb.notna().to_numpy(); assert int(v.sum()) == 29998 and bb.index[~v].tolist() == [125238, 7312] or set(bb.index[~v]) == {125238, 7312}
    ym = y_te[v]; Pm = {"BN": bb.to_numpy(float)[v], "LR": np.asarray(ab["LR_all"], float)[v], "RF": np.asarray(ab["RF_all"], float)[v]}
    c30 = dv.cost(y_te, np.asarray(ab["LR_all"], float) >= 0.69, 5); c29 = dv.cost(ym, Pm["LR"] >= 0.69, 5)
    print("locked LR at tau 0.69: cost on 30,000 rows %d | on the 29,998 BN-defined rows %d (difference %d)" % (c30, c29, c30 - c29))
    Tm = {"BN": {r: 1.0 / (1 + r) for r in RS}, "LR": {r: dv.oof_tau(oof["LR"], y_tr, r, "highest")[0] for r in RS}, "RF": {r: dv.oof_tau(oof["RF"], y_tr, r, "highest")[0] for r in RS}}
    mon = dv.curve(ym, Pm, Tm, RS); mon.insert(0, "contribution", "monitoring"); mon["n_rows"] = len(ym)
    comb = pd.concat([sel, mon], ignore_index=True)

    # ---- summaries
    pd.set_option("display.width", 250, "display.max_columns", 30)
    for c, d in (("selection", sel), ("monitoring", mon)):
        print("\n================ %s (n=%d) ================" % (c, d.n_rows.iloc[0]))
        b = d[d.model == "BN"].set_index("r")
        fl = b.index[b.flagged >= 1]; xo = d[d.model == "BN"].set_index("r").cheaper_policy
        cross = [r for r in RS if xo[r] == "reject_all"]
        print("smallest r at which the BN flags >= 1 test row: %s" % (int(fl.min()) if len(fl) else "none"))
        print("smallest r at which reject-all is the cheaper trivial policy (accept-all stops being cheaper): %s" % (cross[0] if cross else "none up to 20"))
        mx = b.saving.max(); print("r of maximum BN saving: %d (saving %.4f)" % (int(b.saving.idxmax()), mx))
        t = d[d.r.isin(SP)].pivot(index="r", columns="model", values="saving"); print("saving vs cheaper trivial policy:"); print(t.to_string(float_format=lambda x: "%.4f" % x))
        t2 = d[d.r.isin(SP)].pivot(index="r", columns="model", values="cost"); t2["accept_all"] = d[(d.model == "BN") & d.r.isin(SP)].set_index("r").accept_all; t2["reject_all"] = d[(d.model == "BN") & d.r.isin(SP)].set_index("r").reject_all
        print("raw costs:"); print(t2.to_string())
        print("taus at r=5/10/15/20: LR %s | RF %s" % ([d[(d.model == 'LR') & (d.r == r)].tau.iloc[0] for r in SP], [d[(d.model == 'RF') & (d.r == r)].tau.iloc[0] for r in SP]))
        print("models with a flagged count of 0 at r=5/10: %s" % {m: [int(d[(d.model == m) & (d.r == r)].flagged.iloc[0]) for r in (5, 10)] for m in ("BN", "LR", "RF")})
    assert [int(sel[(sel.model == 'BN') & (sel.r == r)].cost.iloc[0]) for r in SP] == [10025, 18069, 21788, 23602]

    # ---- figure
    COL = {"BN": "tab:green", "LR": "tab:blue", "RF": "tab:orange"}
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.4), sharey=False)
    for ax, (c, d) in zip(axes, (("Selection (application stage)", sel), ("Monitoring (full variable set)", mon))):
        for m in ("BN", "LR", "RF"):
            q = d[d.model == m].sort_values("r"); ax.plot(q.r, q.saving * 100, marker="o", ms=3.5, color=COL[m], label=m)
        for r in SP: ax.axvline(r, color="gray", lw=0.7, ls=":")
        ax.axhline(0, color="black", lw=0.8); ax.set_xticks(RS[1::2] if False else [1, 5, 10, 15, 20]); ax.set_xlabel("Cost ratio r (FN cost : FP cost)"); ax.set_ylabel("Saving vs the cheaper trivial policy (%)"); ax.set_title(c)
        ax.grid(False); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    axes[0].legend(frameon=False, loc="upper left")
    fig.suptitle("Decision value across cost ratios: saving vs the cheaper of accept-all and reject-all (test set)")
    cap = ("Saving = 1 - cost / cost of the cheaper trivial policy at that r. BN threshold = 1/(1+r), fixed a priori; LR and RF thresholds are the cost-minimising value on out-of-fold TRAIN predictions at each r (0.01 grid, ties to the highest). "
           "Dotted lines mark the pre-specified r = 5, 10, 15, 20. Monitoring uses the 29,998 test rows where the BN is defined, so LR/RF costs differ slightly from the locked 30,000-row figures. Descriptive only.")
    fig.tight_layout(rect=[0, 0.1, 1, 0.95]); fig.text(0.01, 0.005, textwrap.fill(cap, 170), fontsize=8, va="bottom")
    bio = io.BytesIO(); fig.savefig(bio, format="png", dpi=300, bbox_inches="tight"); plt.close(fig)

    def wr(rel, data):
        p = ss.check_path(os.path.join(R, rel))
        with open(p, "xb") as f: f.write(data)
        assert hashlib.sha256(open(p, "rb").read()).hexdigest() == hashlib.sha256(data).hexdigest(); return hashlib.sha256(data).hexdigest()[:12]
    for k, data in (("sel", sel.to_csv(index=False).encode()), ("mon", mon.to_csv(index=False).encode()), ("comb", comb.to_csv(index=False).encode()), ("fig", bio.getvalue())):
        print("saved %-58s sha12 %s" % (OUT[k], wr(OUT[k], data)))
    print("\nPART B DONE")
_c178()

In [ ]:
# Cell 179 (08_story_extras.ipynb): Part C information ladder (linear LR only; descriptive). Writes 2 files (exclusive create). No restart.
def _c179():
    import os, sys, json, pickle, hashlib, importlib.util, time, warnings
    import numpy as np, pandas as pd
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score, average_precision_score
    from scipy.stats import ks_2samp
    warnings.filterwarnings("ignore")
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sys.path.insert(0, R + "/shared/src"); import safe_save as ss
    def load(name, rel):
        s = importlib.util.spec_from_file_location(name, R + "/" + rel); m = importlib.util.module_from_spec(s); s.loader.exec_module(m); return m
    ck = load("checks", "shared/src/checks.py"); rq3 = load("rq3", "shared/src/rq3.py")
    O_T, O_B = "selection/tables/sel_info_ladder_table.csv", "selection/tables/sel_info_ladder_bootstrap.csv"
    for rel in (O_T, O_B): ss.check_path(os.path.join(R, rel))
    L = lambda p: pickle.load(open(p, "rb")); PROC = R + "/Data/processed/"
    sp = L(PROC + "split_indices.pkl"); y_tr = np.asarray(sp["y_train"]).astype(int); y_te = np.asarray(sp["y_test"]).astype(int)
    g = {"os": os, "np": np, "pd": pd, "L": L, "PROC": PROC, "ROOT": R}
    g["train_idx"], g["test_idx"] = np.asarray(sp["train_idx"]), np.asarray(sp["test_idx"]); g["y_train"], g["y_test"] = sp["y_train"], sp["y_test"]
    code = "".join(json.load(open(R + "/selection/notebooks/00_precheck.ipynb"))["cells"][2]["source"])
    exec(compile(code[:code.index("# ---- validation")], "build_design_src", "exec"), g)
    g["bin_natural_category"] = lambda s, max_category=3: rq3._bin_natural(s, "wrap", max_category)   # 'wrap' binning of rq3 v2 (identical to locked where the locked one works)
    bd = g["build_design"]
    S6 = ['Age', 'Debt Ratio', 'Monthly Income', 'Open Credit Lines', 'Real Estate Loans', 'Number of Dependents']
    RUNGS = {"R0": ['Age', 'Debt Ratio'], "R1": S6, "R2": S6 + ['Credit Utilisation Rate'],
             "R3": S6 + ['30-59 Days Past Due', '90+ Days Late', '60-89 Days Past Due'], "R4": list(g["IMPUTE_ORDER"])}
    DESC = {"R0": "Age + Debt Ratio (additive one-hot bins)", "R1": "six selection variables", "R2": "R1 + Credit Utilisation Rate", "R3": "R1 + the three past-due counts", "R4": "all ten variables (full)"}
    P = {}; ni = {}; t0 = time.time()
    for k, cols in RUNGS.items():
        d = bd(cols); Xtr, Xte = d["Xtr"].values, d["Xte"].values
        for C in (1, 100):
            m = ck.fit_converged(LogisticRegression(C=C, penalty="l2", solver="lbfgs", class_weight="balanced", max_iter=5000), Xtr, y_tr, name="%s C=%d" % (k, C))
            P[(k, C)] = m.predict_proba(Xte)[:, 1]; ni[(k, C)] = int(np.max(m.n_iter_))
        print("  %s done: %d columns, %d variables (%.0fs)" % (k, Xtr.shape[1], len(cols), time.time() - t0))
    auc = {kc: roc_auc_score(y_te, p) for kc, p in P.items()}
    r0 = float(pd.read_csv(R + "/selection/tables/sel_additive_lbfgs_diagnostic.csv")["auc"].iloc[0])
    print("\nchecks: R0 C=1 %.4f vs %.4f | R1 C=1 %.4f vs 0.6903 | R4 C=100 %.4f vs 0.8618" % (auc[("R0", 1)], r0, auc[("R1", 1)], auc[("R4", 100)]))
    assert abs(auc[("R0", 1)] - r0) <= 0.001 and abs(auc[("R1", 1)] - 0.6903) <= 0.001 and abs(auc[("R4", 100)] - 0.8618) <= 0.001, "ladder check failed: STOP, nothing saved"
    n = len(y_te); B = 2000; rng = np.random.RandomState(42); boot = {k: np.empty(B) for k in RUNGS}
    for i in range(B):
        ix = rng.randint(0, n, n); yy = y_te[ix]
        for k in RUNGS: boot[k][i] = roc_auc_score(yy, P[(k, 1)][ix])
    NOTE = "linear LR, not the BN; descriptive; the variable set is fixed on observability grounds and is not changed by these results; C=1 fixed for all rungs (no tuning)"
    rows = []
    for k in RUNGS:
        p = P[(k, 1)]; a = auc[(k, 1)]; lo, hi = np.percentile(boot[k], [2.5, 97.5])
        rows.append(dict(rung=k, description=DESC[k], n_variables=len(RUNGS[k]), auc=a, auc_ci_lo=lo, auc_ci_hi=hi, gini=2 * a - 1, ks=float(ks_2samp(p[y_te == 1], p[y_te == 0]).statistic),
                         ap=float(average_precision_score(y_te, p)), n_iter_C1=ni[(k, 1)], auc_C100_check=auc[(k, 100)], n_iter_C100=ni[(k, 100)], note=NOTE))
    T = pd.DataFrame(rows)
    def comp(lab, a, b):
        d = boot[a] - boot[b]; obs = auc[(a, 1)] - auc[(b, 1)]; lo, hi = np.percentile(d, [2.5, 97.5])
        return dict(comparison=lab, diff=obs, ci_lo=lo, ci_hi=hi, p_value=max(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())), 1 / 2001), n_resamples=B, note=NOTE)
    brow = [comp("%s - %s" % (a, b), a, b) for a, b in (("R1", "R0"), ("R2", "R1"), ("R3", "R1"), ("R4", "R2"), ("R4", "R3"), ("R4", "R1"))]
    dd = (boot["R2"] - boot["R1"]) - (boot["R4"] - boot["R3"]); obs = (auc[("R2", 1)] - auc[("R1", 1)]) - (auc[("R4", 1)] - auc[("R3", 1)]); lo, hi = np.percentile(dd, [2.5, 97.5])
    brow.append(dict(comparison="(R2-R1) minus (R4-R3): value of Credit Utilisation without vs with the counts [extra]", diff=obs, ci_lo=lo, ci_hi=hi,
                     p_value=max(min(1.0, 2 * min((dd <= 0).mean(), (dd >= 0).mean())), 1 / 2001), n_resamples=B, note=NOTE))
    Bt = pd.DataFrame(brow); pd.set_option("display.width", 250, "display.max_columns", 30, "display.max_colwidth", 70)
    print("\nINFORMATION LADDER (test, n=30,000):"); print(T.drop(columns="note").to_string(index=False, float_format=lambda x: "%.4f" % x))
    print("\nPaired bootstrap (2,000, seed 42, shared resamples; p floor 0.0005):"); print(Bt.drop(columns="note").to_string(index=False, float_format=lambda x: "%.4f" % x))
    def wr(rel, data):
        p = ss.check_path(os.path.join(R, rel))
        with open(p, "xb") as f: f.write(data)
        assert hashlib.sha256(open(p, "rb").read()).hexdigest() == hashlib.sha256(data).hexdigest(); return hashlib.sha256(data).hexdigest()[:12]
    print("\nsaved:", wr(O_T, T.to_csv(index=False).encode()), wr(O_B, Bt.to_csv(index=False).encode()))
_c179()

In [ ]:
# Cell 180 (08_story_extras.ipynb): D1 stage agreement + D2 case selection (rules fixed in spec). Writes sel_stage_agreement.csv only. No restart.
def _c180():
    import os, sys, pickle, hashlib, importlib.util, warnings
    import numpy as np, pandas as pd
    warnings.filterwarnings("ignore")
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sys.path.insert(0, R + "/shared/src"); import safe_save as ss
    def load(name, rel):
        s = importlib.util.spec_from_file_location(name, R + "/" + rel); m = importlib.util.module_from_spec(s); s.loader.exec_module(m); return m
    sens = load("sens", "shared/src/sensitivity.py")
    OUT = "selection/tables/sel_stage_agreement.csv"; ss.check_path(os.path.join(R, OUT))
    L = lambda p: pickle.load(open(p, "rb"))
    sb = L(R + "/selection/predictions/sel_bn_test_predictions.pkl"); bb = L(R + "/full_variable/predictions/bn_test_predictions.pkl")["bn_test_probs"]
    v = bb.notna().to_numpy(); idx = bb.index[v]
    assert list(pd.Index(sb["index"])[v]) == list(idx), "row labels differ: STOP"
    ps = pd.Series(np.asarray(sb["p"], float), index=pd.Index(sb["index"])).loc[idx]; pm = bb.loc[idx]
    y = pd.Series(np.asarray(sb["y"]).astype(int), index=pd.Index(sb["index"])).loc[idx]
    print("universe: %d rows, defaulters %d, identical labels asserted" % (len(idx), y.sum()))
    rows = []
    for r in (10, 5):
        t = 1.0 / (1 + r); fs, fm = ps >= t, pm >= t
        for a, b, lab in ((True, True, "yes-yes"), (True, False, "yes-no"), (False, True, "no-yes"), (False, False, "no-no")):
            m = (fs == a) & (fm == b); n = int(m.sum())
            rows.append(dict(r=r, tau=t, flagged_at_application=a, flagged_at_monitoring=b, group=lab, n=n, share_of_rows=n / len(idx), default_rate=(float(y[m].mean()) if n else np.nan),
                             mean_p_selection=(float(ps[m].mean()) if n else np.nan), mean_p_monitoring=(float(pm[m].mean()) if n else np.nan), n_defaulters=int(y[m].sum()),
                             share_of_all_defaulters=float(y[m].sum() / y.sum()), note=("DEGENERATE: selection BN flags nobody at r=5" if r == 5 else "")))
    A = pd.DataFrame(rows); pd.set_option("display.width", 250, "display.max_columns", 30)
    print(A.to_string(index=False, float_format=lambda x: "%.4f" % x))
    # D2 rules at r=10 (tau = 1/11)
    t = 1.0 / 11
    def low_median(s):
        s = s.sort_values(kind="mergesort"); return s.iloc[(len(s) - 1) // 2]
    def pick(sub_vals):  # row at the lower median; ties -> smallest row label
        med = low_median(sub_vals); c = sub_vals[sub_vals == med].index; return int(min(c))
    ya = (ps >= t) & (pm >= t); yb = (ps < t) & (pm >= t)
    caseA = pick(pm[ya]); caseB = pick(pm[yb])
    print("\nRule A (flagged at both stages): %d rows -> row %d" % (ya.sum(), caseA))
    print("Rule B (not flagged at application, flagged at monitoring): %d rows -> row %d" % (yb.sum(), caseB))
    # Rule C
    d = L(R + "/full_variable/models/bn_final_model_iteration79.pkl")
    from pgmpy.models import DiscreteBayesianNetwork
    bn = DiscreteBayesianNetwork([tuple(e) for e in d["model_edges"]]); bn.add_cpds(*d["cpds"])
    FB = ['Credit Utilisation Rate (bin)', 'Age (bin)', 'Debt Ratio (bin)', 'Open Credit Lines (bin)', '30-59 Days Past Due (bin)', '90+ Days Late (bin)', '60-89 Days Past Due (bin)', 'Real Estate Loans (bin)', 'Monthly Income (bin)', 'Number of Dependents (bin)']
    te = L(R + "/Data/processed/df_test.pkl"); mb = ['Credit Utilisation Rate (bin)', '30-59 Days Past Due (bin)', '60-89 Days Past Due (bin)', '90+ Days Late (bin)']
    STATES = {c: bn.get_cpds(c).state_names[c] for c in FB}
    flagged = list(idx[(pm >= t).to_numpy()]); evid = {}
    for i in flagged:
        r_ = te.loc[i]; e = {c: r_[c] for c in FB if pd.notnull(r_[c])}; e["M_Income"] = int(r_["M_Income"]); e["M_Dependents"] = int(r_["M_Dependents"]); evid[i] = e
    res = sens.analyze_blanket(bn, flagged, evid, sorted(mb), STATES, t, verbose=False)
    res["drop"] = res.baseline_prob - res.retracted_prob
    g = res.groupby("row_index").agg(maxdelta=("delta", "max"), anydrop=("drop", lambda s: bool((s > 0).any() and False)))
    below = res[res.retracted_prob < t].groupby("row_index").delta.max()
    maxd = below.rename("maxdelta")
    print("Rule C (flagged at monitoring; some single blanket retraction drops posterior below 1/11): %d rows" % len(maxd))
    caseC = pick(maxd); print("  largest single-variable realised delta (among rows satisfying the rule): median row -> %d (delta %.5f)" % (caseC, maxd.loc[caseC]))
    print("\nSELECTED ROW LABELS (default label displayed only AFTER selection): A=%d B=%d C=%d" % (caseA, caseB, caseC))
    for k, i in (("A", caseA), ("B", caseB), ("C", caseC)):
        print("  case %s row %d | selection p %.4f | monitoring p %.4f | default label (display only) %d" % (k, i, ps[i], pm[i], y[i]))
    data = A.to_csv(index=False).encode(); p = ss.check_path(os.path.join(R, OUT))
    with open(p, "xb") as f: f.write(data)
    print("\nsaved %s sha12 %s" % (OUT, hashlib.sha256(data).hexdigest()[:12]))
_c180()

In [ ]:
# Cell 181 (08_story_extras.ipynb): D3 per-case table + D4 completions with assertions. Writes 3 files (exclusive create) only if all pass. No restart.
def _c181():
    import os, sys, json, pickle, hashlib, importlib.util, warnings
    import numpy as np, pandas as pd
    from pgmpy.models import DiscreteBayesianNetwork
    from pgmpy.inference import VariableElimination
    warnings.filterwarnings("ignore")
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sys.path.insert(0, R + "/shared/src"); import safe_save as ss
    def load(name, rel):
        s = importlib.util.spec_from_file_location(name, R + "/" + rel); m = importlib.util.module_from_spec(s); s.loader.exec_module(m); return m
    sens = load("sens", "shared/src/sensitivity.py")
    O = {"ex": "selection/tables/sel_worked_examples.csv", "co": "selection/tables/sel_worked_example_completions.csv", "nt": "selection/tables/sel_worked_examples_notes.json"}
    for rel in O.values(): ss.check_path(os.path.join(R, rel))
    L = lambda p: pickle.load(open(p, "rb")); T = 1.0 / 11; TOL = 1e-10
    sb = L(R + "/selection/predictions/sel_bn_test_predictions.pkl"); bbs = L(R + "/full_variable/predictions/bn_test_predictions.pkl")["bn_test_probs"]
    v = bbs.notna().to_numpy(); idx = bbs.index[v]
    ps = pd.Series(np.asarray(sb["p"], float), index=pd.Index(sb["index"])).loc[idx]; pm = bbs.loc[idx]
    sn = json.load(open(R + "/Data/processed/sel_state_names.json"))
    raw_s = pd.read_pickle(R + "/Data/processed/sel_test.pkl")
    sel_te = pd.DataFrame({c: (raw_s[c] if set(raw_s[c].dropna().unique()) <= set(sn[c]) else raw_s[c].map(lambda x: str(x) if pd.notnull(x) else np.nan)) for c in raw_s.columns})
    mon_te = L(R + "/Data/processed/df_test.pkl"); clean = L(R + "/Data/processed/df_clean.pkl")
    # networks
    b = L(R + "/selection/models/sel_bn_final.pkl"); bn_s = DiscreteBayesianNetwork([tuple(e) for e in b["model_edges"]]); bn_s.add_cpds(*b["cpds"])
    d = L(R + "/full_variable/models/bn_final_model_iteration79.pkl"); bn_m = DiscreteBayesianNetwork([tuple(e) for e in d["model_edges"]]); bn_m.add_cpds(*d["cpds"])
    FB = ['Credit Utilisation Rate (bin)', 'Age (bin)', 'Debt Ratio (bin)', 'Open Credit Lines (bin)', '30-59 Days Past Due (bin)', '90+ Days Late (bin)', '60-89 Days Past Due (bin)', 'Real Estate Loans (bin)', 'Monthly Income (bin)', 'Number of Dependents (bin)']
    SEL6 = ['Age (bin)', 'Debt Ratio (bin)', 'Monthly Income (bin)', 'Number of Dependents (bin)', 'Real Estate Loans (bin)', 'Open Credit Lines (bin)']
    MB_M = ['Credit Utilisation Rate (bin)', '30-59 Days Past Due (bin)', '60-89 Days Past Due (bin)', '90+ Days Late (bin)']; MB_S = ['Age (bin)', 'Debt Ratio (bin)']
    ST_M = {c: bn_m.get_cpds(c).state_names[c] for c in FB}; ST_S = {c: bn_s.get_cpds(c).state_names[c] for c in SEL6}
    def ev_m(i):
        r = mon_te.loc[i]; e = {c: r[c] for c in FB if pd.notnull(r[c])}; e["M_Income"] = int(r["M_Income"]); e["M_Dependents"] = int(r["M_Dependents"]); return e
    def ev_s(i):
        r = sel_te.loc[i]; e = {c: r[c] for c in SEL6 if pd.notnull(r[c])}; e["M_Income"] = int(r["M_Income"]); e["M_Dependents"] = int(r["M_Dependents"]); return e

    # ---- re-derive the cases and the Rule C alternative
    def low_median(s):
        s = s.sort_values(kind="mergesort"); return s.iloc[(len(s) - 1) // 2]
    def pick(s):
        m = low_median(s); return int(min(s[s == m].index))
    ya = (ps >= T) & (pm >= T); yb = (ps < T) & (pm >= T); caseA, caseB = pick(pm[ya]), pick(pm[yb])
    flagged = list(idx[(pm >= T).to_numpy()]); evm = {i: ev_m(i) for i in flagged}
    res = sens.analyze_blanket(bn_m, flagged, evm, sorted(MB_M), ST_M, T, verbose=False)
    maxd_cross = res[res.retracted_prob < T].groupby("row_index").delta.max()
    caseC = pick(maxd_cross)
    maxd_all = res[res.row_index.isin(maxd_cross.index)].groupby("row_index").delta.max(); alt = pick(maxd_all)
    print("cases re-derived: A=%d B=%d C=%d | alternative Rule C (largest delta across all four, same row set) would pick %d (record only)" % (caseA, caseB, caseC, alt))
    assert (caseA, caseB, caseC) == (607, 1561, 1187), "cases differ from the selected ones: STOP"
    CASES = {"A": caseA, "B": caseB, "C": caseC}

    # ---- bin recomputation from raw values with the saved edges
    pa = L(R + "/Data/processed/preprocessing_artifacts.pkl")["bin_edges"]
    def cut(x, e):
        if pd.isnull(x): return None
        ea = np.array(e, float).copy(); ea[0] = -np.inf; ea[-1] = np.inf
        return str(pd.cut(pd.Series([x]), bins=ea, include_lowest=True).iloc[0])
    def natural(x):
        if pd.isnull(x): return None
        lab = ["0", "1", "2", "3+"]; return lab[int(x)] if x < 3 else "3+"
    EDGE_VARS = ['Credit Utilisation Rate', 'Age', 'Debt Ratio', 'Open Credit Lines', 'Monthly Income']
    RAW = ['Credit Utilisation Rate', 'Age', 'Debt Ratio', 'Monthly Income', 'Open Credit Lines', '30-59 Days Past Due', '90+ Days Late', '60-89 Days Past Due', 'Number of Dependents', 'Real Estate Loans']
    def sbin(x): return "missing" if pd.isnull(x) else str(x)
    for k, i in CASES.items():
        assert i in clean.index and i in sel_te.index and i in mon_te.index, "row label missing in a source: STOP"
        for c in RAW:
            rb = cut(clean.at[i, c], pa[c]) if c in EDGE_VARS else natural(clean.at[i, c])
            sm = mon_te.at[i, c + " (bin)"]; ss_ = sel_te.at[i, c + " (bin)"] if c + " (bin)" in sel_te.columns else None
            exp = "missing" if rb is None else rb
            assert sbin(sm) == exp, "monitoring bin differs from recomputed for row %d %s: %s vs %s" % (i, c, sbin(sm), exp)
            if ss_ is not None: assert sbin(ss_) == exp, "selection bin differs for row %d %s" % (i, c)
        assert int(clean.at[i, "Default"]) == int(mon_te.at[i, "Default"]) == int(sel_te.at[i, "Default"])
    print("row labels identical across df_clean/sel_test/df_test and every bin recomputed from raw values with the saved edges: PASS")

    # ---- baselines
    for i in CASES.values():
        assert abs(pm[i] - ps_ if False else 0) == 0 if False else True
    eng_s, eng_m = VariableElimination(bn_s), VariableElimination(bn_m)
    def post(eng, e):
        try: p = float(eng.query(variables=["Default"], evidence=e, show_progress=False).get_value(Default=1))
        except Exception: p = np.nan
        return p if np.isfinite(p) else np.nan
    rows_ex, rows_co = [], []; nassert = 0; rep = []
    STAGES = {"selection": (bn_s, eng_s, ev_s, MB_S, ST_S, ps), "monitoring": (bn_m, eng_m, ev_m, MB_M, ST_M, pm)}
    for case, i in CASES.items():
        for stage, (bn, eng, evf, MB, ST, pser) in STAGES.items():
            e = evf(i); base = post(eng, e); assert abs(base - float(pser[i])) <= TOL, "baseline differs from saved predictions (%s row %d): STOP" % (stage, i)
            ana = sens.analyze_blanket(bn, [i], {i: e}, sorted(MB), ST, T, skip_unobserved=True, verbose=False).set_index("variable")
            for X in sorted(MB):
                er = {k: v_ for k, v_ in e.items() if k != X}; pr = post(eng, er)
                f = eng.query(variables=[X], evidence=er, show_progress=False); w = dict(zip(f.state_names[X], np.asarray(f.values, float)))
                comp = []
                for x in ST[X]:
                    px = post(eng, {**er, X: x}); comp.append((x, w[x], px, x == e[X]))
                    rows_co.append(dict(case=case, row_label=i, stage=stage, variable=X.replace(" (bin)", ""), x=str(x), w_x=w[x], p_x=px, is_actual=(x == e[X]), p_retracted=pr, baseline=base, threshold=T))
                ws = np.array([c[1] for c in comp]); px = np.array([c[2] for c in comp]); pos = ws > 0
                assert abs(ws.sum() - 1) <= TOL, "(i) sum w != 1: %s %s %d" % (stage, X, i)
                assert np.isfinite(px[pos]).all(), "p_x undefined where w_x > 0: %s %s %d" % (stage, X, i)
                assert abs((ws[pos] * px[pos]).sum() - pr) <= TOL, "(ii) law of total probability fails: %s %s %d" % (stage, X, i)
                act = [c for c in comp if c[3]][0]; assert abs(act[2] - base) <= TOL, "(iii) p_x at actual != baseline: %s %s %d" % (stage, X, i)
                sexp_calc = float((ws[pos] * np.abs(px[pos] - base)).sum()); sexp = float(ana.loc[X, "S_exp"])
                assert abs(sexp_calc - sexp) <= TOL, "STOP: sum w|p_x-baseline| != validated S_exp (%s %s %d): %.3e vs %.3e" % (stage, X, i, sexp_calc, sexp)
                nassert += 4; sret = float((ws[pos] * np.abs(px[pos] - pr)).sum())
                dec = lambda p: p >= T
                pfs = float(sum(c[1] for c in comp if c[1] > 0 and dec(c[2]) != dec(pr)))
                rep.append((case, i, stage, X.replace(" (bin)", ""), sret - sexp))
                rows_ex.append(dict(case=case, row_label=i, stage=stage, variable=X.replace(" (bin)", ""), default_label_display_only=int(clean.at[i, "Default"]),
                                    p_selection=float(ps[i]), selection_cell="%s x %s" % (sel_te.at[i, "Age (bin)"], sel_te.at[i, "Debt Ratio (bin)"]), p_monitoring=float(pm[i]),
                                    stage_baseline=base, S_exp=sexp, p_retracted=pr, realised_flip=bool(dec(base) != dec(pr)), prospective_flip_share=pfs,
                                    sum_w_abs_px_minus_baseline=sexp_calc, sum_w_abs_px_minus_pretracted=sret, threshold=T,
                                    **{"raw_" + c: ("missing" if pd.isnull(clean.at[i, c]) else float(clean.at[i, c])) for c in RAW},
                                    **{"bin_" + c: sbin(mon_te.at[i, c + " (bin)"]) for c in RAW}))
    print("D4 assertions: %d checks passed ((i) sum w = 1, (ii) law of total probability, (iii) p_x at actual = baseline, sum w|p_x - baseline| = validated S_exp), all to 1e-10" % nassert)
    EX, CO = pd.DataFrame(rows_ex), pd.DataFrame(rows_co); pd.set_option("display.width", 250, "display.max_columns", 40)
    print("\nPER-CASE TABLE (case, stage, variable):"); print(EX[["case", "row_label", "stage", "variable", "default_label_display_only", "stage_baseline", "S_exp", "p_retracted", "realised_flip", "prospective_flip_share", "sum_w_abs_px_minus_pretracted"]].to_string(index=False, float_format=lambda x: "%.5f" % x))
    print("\nreported only: sum w|p_x - p_retracted| minus S_exp, range %.4f to %.4f" % (min(r[4] for r in rep), max(r[4] for r in rep)))
    for k, i in CASES.items(): print("case %s row %d: raw %s | selection cell %s" % (k, i, {c: ("missing" if pd.isnull(clean.at[i, c]) else round(float(clean.at[i, c]), 4)) for c in RAW}, EX[EX.case == k].selection_cell.iloc[0]))
    notes = dict(rules="A: flagged both stages, lower median of monitoring p; B: not flagged at application, flagged at monitoring, lower median monitoring p; C: flagged at monitoring where a single blanket retraction drops p below 1/11, row at the lower median of each row's largest delta among crossing retractions; ties to the smallest row label; default label never used for selection",
                 rows_satisfying=dict(A=int(ya.sum()), B=int(yb.sum()), C=int(len(maxd_cross))), cases=CASES, threshold="1/11 (r=10)",
                 rule_C_alternative_for_record_only=dict(description="largest delta across all four variables among the same row set", would_pick=int(alt), used=False, no_figure_or_table_built_from_it=True),
                 oof_tie_rule="Part B decision-value curves use ties-to-highest-tau on out-of-fold train predictions; the Phase 5 value 0.90 (LR, r=5) was lowest-tie (plateau 0.90-0.99; both flag 0 test rows, same test cost)",
                 oof_imputation_caveat="monitoring OOF predictions use the locked imputer fitted on ALL train rows (feature columns only, no labels), so they are not fully out-of-sample with respect to imputation",
                 d4_assertions="(i) sum w = 1; (ii) sum w*p_x = p_retracted; (iii) p_x at actual = baseline; sum w|p_x - baseline| = validated S_exp; all to 1e-10",
                 reported_not_asserted="sum w|p_x - p_retracted|", footer="sensitivity to unseen evidence, not the reason for a decision; thresholds 1/11 (r=10)")
    def wr(rel, data):
        p = ss.check_path(os.path.join(R, rel))
        with open(p, "xb") as f: f.write(data)
        assert hashlib.sha256(open(p, "rb").read()).hexdigest() == hashlib.sha256(data).hexdigest(); return hashlib.sha256(data).hexdigest()[:12]
    print("\nsaved:", wr(O["ex"], EX.to_csv(index=False).encode()), wr(O["co"], CO.to_csv(index=False).encode()), wr(O["nt"], json.dumps(notes, indent=1, default=str).encode()))
_c181()

In [ ]:
# Cell 182 (08_story_extras.ipynb): Part D2 figures. Reads saved tables; writes 4 PNGs (exclusive create). No restart.
def _c182():
    import os, sys, io, hashlib, warnings
    import numpy as np, pandas as pd
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    from matplotlib.patches import PathPatch, Rectangle
    from matplotlib.path import Path
    warnings.filterwarnings("ignore")
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sys.path.insert(0, R + "/shared/src"); import safe_save as ss
    F = "selection/figures/"; NM = ["sel_worked_fan.png", "sel_worked_journey.png", "sel_stage_flow.png", "sel_worked_age_bubbles.png"]
    for n in NM: ss.check_path(os.path.join(R, F + n))
    T = 1 / 11; G = "tab:green"; FOOT = "sensitivity to unseen evidence, not the reason for a decision; thresholds 1/11 (r=10)"
    ex = pd.read_csv(R + "/selection/tables/sel_worked_examples.csv"); co = pd.read_csv(R + "/selection/tables/sel_worked_example_completions.csv")
    ag = pd.read_csv(R + "/selection/tables/sel_stage_agreement.csv"); cells = pd.read_csv(R + "/selection/tables/sel_default_cells.csv")
    def fin(fig, extra=""):
        fig.text(0.01, 0.005, (extra + " " if extra else "") + FOOT, fontsize=8, va="bottom"); b = io.BytesIO(); fig.savefig(b, format="png", dpi=300, bbox_inches="tight"); plt.close(fig); return b.getvalue()
    def clean(ax):
        ax.grid(False); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    out = {}
    CASES = ["A", "B", "C"]; TITLE = {"A": "A: stable flag", "B": "B: late-revealed risk", "C": "C: decision rests on unseen evidence"}
    # 1 fan
    panels = [(c, s, v) for c in CASES for s in ("selection", "monitoring") for v in (["Age", "Debt Ratio"] if s == "selection" else ["Credit Utilisation Rate", "30-59 Days Past Due", "60-89 Days Past Due", "90+ Days Late"])]
    ncol = 6; nrow = len(CASES) * 2
    fig, axes = plt.subplots(nrow, ncol, figsize=(22, 13)); axes = axes.reshape(nrow, ncol)
    for a in axes.ravel(): a.axis("off")
    for ci, c in enumerate(CASES):
        for si, s in enumerate(("selection", "monitoring")):
            vs = ["Age", "Debt Ratio"] if s == "selection" else ["Credit Utilisation Rate", "30-59 Days Past Due", "60-89 Days Past Due", "90+ Days Late"]
            for vi, v in enumerate(vs):
                ax = axes[ci * 2 + si, vi]; ax.axis("on"); d = co[(co.case == c) & (co.stage == s) & (co.variable == v)]; d = d[d.w_x > 0]
                pr = d.p_retracted.iloc[0]; base = d.baseline.iloc[0]
                ax.scatter(d.p_x, np.zeros(len(d)), s=d.w_x * 2500 + 6, color=G, alpha=0.5, edgecolor="none")
                a = d[d.is_actual]; ax.scatter(a.p_x, np.zeros(len(a)), s=a.w_x * 2500 + 6, facecolor="none", edgecolor="black", lw=1.8)
                ax.axvline(T, color="black", ls="--", lw=0.9); ax.scatter([pr], [0], marker="D", color="tab:red", zorder=5, s=36)
                f = ex[(ex.case == c) & (ex.stage == s) & (ex.variable == v)].prospective_flip_share.iloc[0]
                ax.set_ylim(-1, 1); ax.set_yticks([]); ax.set_title("%s | %s\n%s" % (TITLE[c][:1], s[:3] + ". stage", v), fontsize=8); ax.set_xlabel("P(Default=1)", fontsize=8); ax.tick_params(labelsize=7)
                ax.text(0.02, 0.9, "prospective flip share %.2f" % f, transform=ax.transAxes, fontsize=7); clean(ax); ax.spines["left"].set_visible(False)
    fig.suptitle("Worked examples: every possible value of each blanket variable (bubble area = P(X=x | other evidence)); ringed = observed value; diamond = posterior without the variable; dashed = 1/11", fontsize=11)
    fig.tight_layout(rect=[0, 0.03, 1, 0.96]); out[NM[0]] = fin(fig, "Rows: case A application, A monitoring, B application, B monitoring, C application, C monitoring.")
    # 2 journey
    fig, axes = plt.subplots(3, 2, figsize=(13, 11), gridspec_kw=dict(width_ratios=[1, 1.3]))
    ages = list(dict.fromkeys(cells.age_bin)); drs = list(dict.fromkeys(cells.debt_ratio_bin))
    M = cells.pivot(index="age_bin", columns="debt_ratio_bin", values="p_default_em").reindex(index=ages, columns=drs)
    for k, c in enumerate(CASES):
        e = ex[(ex.case == c) & (ex.stage == "selection")].iloc[0]; ageb, drb = e.selection_cell.split(" x ")
        ax = axes[k, 0]; ax.imshow(M.values, cmap="YlOrRd", vmin=0, vmax=0.18, aspect="auto")
        for i in range(5):
            for j in range(5): ax.text(j, i, "%.3f" % M.values[i, j], ha="center", va="center", fontsize=6)
        ri, rj = ages.index(ageb), drs.index(drb); ax.add_patch(Rectangle((rj - .5, ri - .5), 1, 1, fill=False, ec="black", lw=2.5))
        ax.set_xticks(range(5)); ax.set_xticklabels(drs, fontsize=6, rotation=20); ax.set_yticks(range(5)); ax.set_yticklabels(ages, fontsize=6); ax.set_title("%s: application stage, cell P = %.4f (threshold 1/11 = 0.0909)" % (TITLE[c], e.p_selection), fontsize=8)
        ax = axes[k, 1]; m = ex[(ex.case == c) & (ex.stage == "monitoring")]
        ax.axvline(T, color="black", ls="--", lw=1); ax.scatter([m.p_monitoring.iloc[0]], [0], s=140, color=G, zorder=4, label="monitoring posterior")
        ax.scatter(m.p_retracted, np.zeros(len(m)), marker="|", s=600, color="tab:red", zorder=3, label="posterior without one variable")
        for _, r in m.iterrows(): ax.text(r.p_retracted, 0.18, r.variable.split()[0], fontsize=7, rotation=45, ha="left")
        ax.set_xlim(0, 0.45); ax.set_ylim(-0.4, 0.8); ax.set_yticks([]); ax.set_xlabel("P(Default=1)"); ax.set_title("%s: monitoring stage, P = %.4f" % (TITLE[c][:1], m.p_monitoring.iloc[0]), fontsize=8); clean(ax); ax.spines["left"].set_visible(False)
    axes[0, 1].legend(frameon=False, fontsize=7, loc="upper right"); fig.suptitle("Worked examples: application cell (ringed) and monitoring posterior on one probability ruler"); fig.tight_layout(rect=[0, 0.03, 1, 0.96]); out[NM[1]] = fin(fig)
    # 3 flow
    a10 = ag[ag.r == 10]; fig, ax = plt.subplots(figsize=(9, 7)); ax.axis("off"); N = a10.n.sum()
    colors = plt.cm.YlOrRd; mx = float(a10.default_rate.max())
    def stack(flag_col):
        y0 = {}; pos = 0.0
        for fl in (True, False):
            tot = a10[a10[flag_col] == fl].n.sum() / N; y0[fl] = (pos, pos + tot); pos += tot + 0.04
        return y0
    L = stack("flagged_at_application"); Rr = stack("flagged_at_monitoring"); cl = {True: 0.0, False: 0.0}; cr = {True: 0.0, False: 0.0}
    for _, r in a10.sort_values(["flagged_at_application", "flagged_at_monitoring"], ascending=False).iterrows():
        h = r.n / N; la, lb = L[r.flagged_at_application][0] + cl[r.flagged_at_application], Rr[r.flagged_at_monitoring][0] + cr[r.flagged_at_monitoring]
        cl[r.flagged_at_application] += h; cr[r.flagged_at_monitoring] += h
        v = [(0.15, la), (0.5, la), (0.5, lb), (0.85, lb), (0.85, lb + h), (0.5, lb + h), (0.5, la + h), (0.15, la + h), (0.15, la)]
        cp = [Path.MOVETO, Path.CURVE4, Path.CURVE4, Path.CURVE4, Path.LINETO, Path.CURVE4, Path.CURVE4, Path.CURVE4, Path.CLOSEPOLY]
        v2 = [(0.15, la), (0.5, la), (0.5, lb), (0.85, lb), (0.85, lb + h), (0.5, lb + h), (0.5, la + h), (0.15, la + h), (0.15, la)]
        ax.add_patch(PathPatch(Path(v2, cp), fc=colors(r.default_rate / mx), ec="gray", lw=0.5, alpha=0.9))
        ax.text(0.5, (la + lb) / 2 + h / 2, "%s\n%s rows, default %.1f%%" % (r.group, format(int(r.n), ","), 100 * r.default_rate), ha="center", va="center", fontsize=7)
    for fl, (a, b) in L.items(): ax.add_patch(Rectangle((0.1, a), 0.05, b - a, color="gray")); ax.text(0.09, (a + b) / 2, "flagged at application" if fl else "not flagged", ha="right", va="center", fontsize=8)
    for fl, (a, b) in Rr.items(): ax.add_patch(Rectangle((0.85, a), 0.05, b - a, color="gray")); ax.text(0.91, (a + b) / 2, "flagged at monitoring" if fl else "not flagged", ha="left", va="center", fontsize=8)
    ax.set_xlim(0, 1); ax.set_ylim(1.1, -0.02); ax.set_title("Stage flow at r = 10: application flag to monitoring flag (band height = share of 29,998 rows, colour = default rate)", fontsize=10)
    out[NM[2]] = fin(fig, "Bezier bands from the stage-agreement table at r=10.")
    # 4 age bubbles
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
    for ax, c in zip(axes, CASES):
        e = ex[(ex.case == c) & (ex.stage == "selection")].iloc[0]; drb = e.selection_cell.split(" x ")[1]; ageb = e.selection_cell.split(" x ")[0]
        col = [M.loc[a, drb] for a in ages]; ax.scatter(range(5), np.zeros(5), s=1500, c=col, cmap="YlOrRd", vmin=0, vmax=0.18, marker="s", edgecolor="gray")
        w = co[(co.case == c) & (co.stage == "selection") & (co.variable == "Age")].set_index("x").w_x
        for i, a in enumerate(ages): ax.scatter([i], [0], s=w.get(a, 0) * 2500, color=G, alpha=0.7); ax.text(i, -0.55, "%.3f\nw=%.2f" % (M.loc[a, drb], w.get(a, 0)), ha="center", fontsize=7)
        ax.scatter([ages.index(ageb)], [0], s=2000, facecolor="none", edgecolor="black", lw=2.5)
        ax.set_xticks(range(5)); ax.set_xticklabels(ages, fontsize=7, rotation=20); ax.set_yticks([]); ax.set_ylim(-1, 0.7); ax.set_title("%s, Debt Ratio %s" % (TITLE[c], drb), fontsize=8); clean(ax); ax.spines["left"].set_visible(False)
    fig.suptitle("Application stage: the borrower's Debt Ratio row, five Age cells (cell colour = P(Default=1); green bubble area = P(Age=x | other evidence); ringed = actual Age)", fontsize=10); fig.tight_layout(rect=[0, 0.05, 1, 0.9]); out[NM[3]] = fin(fig)
    for n, b in out.items():
        p = ss.check_path(os.path.join(R, F + n))
        with open(p, "xb") as f: f.write(b)
        assert hashlib.sha256(open(p, "rb").read()).hexdigest() == hashlib.sha256(b).hexdigest(); print("saved %-32s %7d bytes sha12 %s" % (n, len(b), hashlib.sha256(b).hexdigest()[:12]))
_c182()

In [ ]:
# Cell 183 (08_story_extras.ipynb): redesigned figures (v2). Reads saved tables; writes 4 PNGs (exclusive create). No restart.
def _c183():
    import os, sys, io, hashlib, warnings
    import numpy as np, pandas as pd
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle, FancyArrowPatch
    warnings.filterwarnings("ignore")
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sys.path.insert(0, R + "/shared/src"); import safe_save as ss
    F = "selection/figures/"; NM = ["sel_stage_flow_v2.png", "sel_story_case_A_v2.png", "sel_story_case_B_v2.png", "sel_story_case_C_v2.png"]
    for n in NM: ss.check_path(os.path.join(R, F + n))
    T = 1 / 11; BLUE, GREEN, FLIP, GREY = "#0072B2", "tab:green", "#D55E00", "#8c8c8c"
    FOOT = "sensitivity to unseen evidence, not the reason for a decision; thresholds 1/11 (r=10)"
    ex = pd.read_csv(R + "/selection/tables/sel_worked_examples.csv"); ag = pd.read_csv(R + "/selection/tables/sel_stage_agreement.csv")
    def fin(fig, note=""):
        fig.text(0.01, 0.005, (note + " " if note else "") + FOOT, fontsize=8, va="bottom"); b = io.BytesIO(); fig.savefig(b, format="png", dpi=300, bbox_inches="tight"); plt.close(fig); return b.getvalue()
    out = {}
    # ---- stage flow as 2x2 tiles
    a = ag[ag.r == 10].set_index("group"); N = int(a.n.sum()); mx = float(a.default_rate.max()); cm = plt.cm.Blues
    LAB = {"yes-yes": "Flagged at both stages", "yes-no": "Flagged at application only", "no-yes": "Flagged at monitoring only", "no-no": "Never flagged"}
    POS = {"yes-yes": (0, 1), "yes-no": (1, 1), "no-yes": (0, 0), "no-no": (1, 0)}
    fig, ax = plt.subplots(figsize=(10, 7.2)); ax.set_xlim(0, 2); ax.set_ylim(0, 2); ax.axis("off")
    for g, (cx, cy) in POS.items():
        r = a.loc[g]; col = cm(0.15 + 0.8 * r.default_rate / mx); dark = r.default_rate / mx > 0.55; tc = "white" if dark else "black"
        ax.add_patch(Rectangle((cx + 0.02, cy + 0.02), 0.96, 0.96, fc=col, ec="white", lw=2))
        ax.text(cx + 0.5, cy + 0.80, LAB[g], ha="center", va="center", fontsize=13, fontweight="bold", color=tc)
        ax.text(cx + 0.5, cy + 0.55, format(int(r.n), ","), ha="center", va="center", fontsize=30, color=tc)
        ax.text(cx + 0.5, cy + 0.33, "borrowers (%.1f%% of all)" % (100 * r.share_of_rows), ha="center", va="center", fontsize=11, color=tc)
        ax.text(cx + 0.5, cy + 0.15, "%.1f%% of them defaulted" % (100 * r.default_rate), ha="center", va="center", fontsize=12, fontweight="bold", color=tc)
    ax.text(0.5, 2.06, "Flagged at monitoring", ha="center", fontsize=12); ax.text(1.5, 2.06, "Not flagged at monitoring", ha="center", fontsize=12)
    ax.text(-0.04, 1.5, "Flagged at\napplication", ha="right", va="center", fontsize=12); ax.text(-0.04, 0.5, "Not flagged at\napplication", ha="right", va="center", fontsize=12)
    ax.set_ylim(0, 2.15); ax.set_xlim(-0.4, 2)
    fig.suptitle("Who is flagged at each stage? %s borrowers, flag threshold 1/11 (r = 10)" % format(N, ","), fontsize=13)
    out[NM[0]] = fin(fig, "Darker tile = higher share of borrowers who defaulted. Of all 2,003 defaulters: 29.6% were missed at application but flagged at monitoring.")
    # ---- story cards
    TITLE = {"A": "Case A: flagged at both stages", "B": "Case B: looked safe at application, flagged at monitoring", "C": "Case C: flagged, but the decision rests on one missing fact"}
    NICE = {"Age": "Age", "Debt Ratio": "Debt Ratio", "Credit Utilisation Rate": "Credit utilisation", "30-59 Days Past Due": "30-59 days past due", "60-89 Days Past Due": "60-89 days past due", "90+ Days Late": "90+ days late"}
    sentences = {}
    for k, case in enumerate(("A", "B", "C")):
        d = ex[ex.case == case]; r0 = d.iloc[0]
        sel = d[d.stage == "selection"].sort_values("S_exp", ascending=False); mon = d[d.stage == "monitoring"].sort_values("S_exp", ascending=False)
        ps, pm = float(r0.p_selection), float(r0.p_monitoring)
        flips = d[d.realised_flip]
        if len(flips):
            fl = "; ".join("without %s the risk would be %.3f and the decision would change (%s stage)" % (NICE[x.variable].lower(), x.p_retracted, x.stage[:3] + ".") for x in flips.itertuples())
        else:
            fl = "no single missing fact would change the decision"
        s = "Application-stage risk %.3f (%s the 0.091 line); monitoring-stage risk %.3f (%s). %s." % (ps, "above" if ps >= T else "below", pm, "above" if pm >= T else "below", fl[0].upper() + fl[1:])
        sentences[case] = s
        rows = [("APPLICATION STAGE", None)] + [(x.variable, x) for x in sel.itertuples()] + [("MONITORING STAGE", None)] + [(x.variable, x) for x in mon.itertuples()]
        n = len(rows); fig, ax = plt.subplots(figsize=(12, 7.4)); xmax = max(0.36, float(max(d.p_retracted.max(), pm, ps)) + 0.03)
        for i, (name, x) in enumerate(rows):
            y = n - i
            if x is None:
                ax.text(-0.005, y, name, ha="right", va="center", fontsize=11, fontweight="bold", color=BLUE if name.startswith("APP") else GREEN); continue
            base = ps if x.stage == "selection" else pm; col = BLUE if x.stage == "selection" else GREEN; flip = bool(x.realised_flip)
            ax.text(-0.005, y, NICE[name], ha="right", va="center", fontsize=11)
            ax.scatter([base], [y], s=120, color=col, zorder=4)
            ax.add_patch(FancyArrowPatch((base, y), (x.p_retracted, y), arrowstyle="-|>", mutation_scale=16, lw=2.6, color=FLIP if flip else GREY, zorder=3, shrinkA=6, shrinkB=0))
            ax.text(xmax + 0.012, y, "%s | flip chance across plausible values: %.0f%%" % ("DECISION FLIPS" if flip else "no flip", 100 * x.prospective_flip_share), va="center", fontsize=9.5, color=FLIP if flip else "black", fontweight="bold" if flip else "normal")
            ax.text((base + x.p_retracted) / 2, y + 0.33, "%.3f" % x.p_retracted, ha="center", fontsize=8, color="dimgray")
        ax.axvline(T, color="black", ls="--", lw=1.2); ax.text(T, n + 0.9, "flag line 1/11 = 0.091", ha="center", fontsize=10)
        ax.set_xlim(0, xmax); ax.set_ylim(0.2, n + 1.3); ax.set_yticks([]); ax.set_xlabel("Estimated probability of default"); ax.grid(False)
        for sp in ("top", "right", "left"): ax.spines[sp].set_visible(False)
        ax.scatter([], [], color=BLUE, s=80, label="risk at application (dot), arrow = risk if this fact were unknown"); ax.scatter([], [], color=GREEN, s=80, label="risk at monitoring (dot)")
        ax.plot([], [], color=FLIP, lw=3, label="arrow crosses the flag line: decision would change"); ax.plot([], [], color=GREY, lw=3, label="decision unchanged")
        ax.legend(loc="lower right", frameon=False, fontsize=9)
        fig.suptitle(TITLE[case], fontsize=14, y=0.99); fig.text(0.5, 0.925, s, ha="center", va="top", fontsize=9.5, wrap=True)
        fig.subplots_adjust(left=0.2, right=0.62, top=0.86, bottom=0.12)
        out[NM[1 + k]] = fin(fig)
        print("case %s sentence: %s" % (case, s))
    for n_, b in out.items():
        p = ss.check_path(os.path.join(R, F + n_))
        with open(p, "xb") as f: f.write(b)
        assert hashlib.sha256(open(p, "rb").read()).hexdigest() == hashlib.sha256(b).hexdigest(); print("saved %-26s %7d bytes sha12 %s" % (n_, len(b), hashlib.sha256(b).hexdigest()[:12]))
_c183()

In [ ]:
# Cell 184 (08_story_extras.ipynb): simplified story cards (v3). Reads saved tables; writes 3 PNGs (exclusive create). No restart.
def _c184():
    import os, sys, io, hashlib, warnings
    import numpy as np, pandas as pd
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    warnings.filterwarnings("ignore")
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sys.path.insert(0, R + "/shared/src"); import safe_save as ss
    F = "selection/figures/"; NM = {c: "sel_story_case_%s_v3.png" % c for c in "ABC"}
    for n in NM.values(): ss.check_path(os.path.join(R, F + n))
    T = 1 / 11; BLUE, GREEN, FLIP, GREY = "#0072B2", "tab:green", "#D55E00", "#8c8c8c"
    FOOT = "sensitivity to unseen evidence, not the reason for a decision; thresholds 1/11 (r=10)"
    ex = pd.read_csv(R + "/selection/tables/sel_worked_examples.csv")
    NICE = {"Age": "Age", "Debt Ratio": "Debt Ratio", "Credit Utilisation Rate": "Credit utilisation", "30-59 Days Past Due": "30-59 days past due", "60-89 Days Past Due": "60-89 days past due", "90+ Days Late": "90+ days late"}
    TITLE = {"A": "Case A: flagged at both stages", "B": "Case B: looked safe at application, flagged at monitoring", "C": "Case C: flagged, but the decision rests on one unseen fact"}
    out = {}
    for case in "ABC":
        d = ex[ex.case == case]; ps, pm = float(d.p_selection.iloc[0]), float(d.p_monitoring.iloc[0])
        sel = d[d.stage == "selection"].sort_values("S_exp", ascending=False); mon = d[d.stage == "monitoring"].sort_values("S_exp", ascending=False)
        rows = [("Application stage", None)] + [(x.variable, x) for x in sel.itertuples()] + [("Monitoring stage", None)] + [(x.variable, x) for x in mon.itertuples()]
        n = len(rows); xmax = max(0.36, float(max(d.p_retracted.max(), ps, pm)) + 0.04)
        fig = plt.figure(figsize=(11, 6.6)); ax = fig.add_axes([0.24, 0.20, 0.56, 0.54])
        for i, (name, x) in enumerate(rows):
            y = n - i
            if x is None:
                ax.text(-0.02, y, name, transform=ax.get_yaxis_transform(), ha="right", va="center", fontsize=12, fontweight="bold", color=BLUE if name.startswith("App") else GREEN); continue
            base = ps if x.stage == "selection" else pm; col = BLUE if x.stage == "selection" else GREEN; flip = bool(x.realised_flip)
            ax.text(-0.02, y, NICE[name], transform=ax.get_yaxis_transform(), ha="right", va="center", fontsize=11)
            ax.plot([base, x.p_retracted], [y, y], color=FLIP if flip else GREY, lw=2.2, zorder=2)
            ax.scatter([x.p_retracted], [y], s=130, facecolor="white", edgecolor=FLIP if flip else GREY, linewidth=2.2, zorder=3)
            ax.scatter([base], [y], s=130, color=col, zorder=4)
            side = 1 if x.p_retracted >= base else -1
            ax.text(x.p_retracted + side * 0.012, y, "%.3f" % x.p_retracted, ha="left" if side > 0 else "right", va="center", fontsize=9.5, color="dimgray")
            if flip: ax.text(1.03, y, "DECISION CHANGES", transform=ax.get_yaxis_transform(), va="center", fontsize=11, fontweight="bold", color=FLIP)
        ax.axvline(T, color="black", ls="--", lw=1.2); ax.text(T, 1.04, "flag line 0.091", transform=ax.get_xaxis_transform(), ha="center", va="bottom", fontsize=10)
        ax.set_xlim(0, xmax); ax.set_ylim(0.3, n + 0.7); ax.set_yticks([]); ax.set_xlabel("Estimated probability of default", fontsize=11); ax.grid(False)
        for sp in ("top", "right", "left"): ax.spines[sp].set_visible(False)
        fig.legend(handles=[Line2D([0], [0], marker="o", color="w", markerfacecolor="dimgray", markersize=10, label="filled dot: estimated risk with all facts known"),
                            Line2D([0], [0], marker="o", color="w", markerfacecolor="white", markeredgecolor="dimgray", markeredgewidth=2, markersize=10, label="open circle: risk if that one fact were unknown"),
                            Line2D([0], [0], color=FLIP, lw=3, label="vermilion: that would change the decision")],
                   loc="lower center", ncol=3, frameon=False, fontsize=9, bbox_to_anchor=(0.5, 0.045))
        flips = d[d.realised_flip]
        l1 = "Risk at application: %.3f (%s).  Risk at monitoring: %.3f (%s)." % (ps, "flagged" if ps >= T else "not flagged", pm, "flagged" if pm >= T else "not flagged")
        l2 = ("Decision would change if we did not know: " + ", ".join("%s (%s)" % (NICE[x.variable].lower(), "application" if x.stage == "selection" else "monitoring") for x in flips.itertuples()) + ".") if len(flips) else "No single unseen fact would change the decision."
        fig.suptitle(TITLE[case], fontsize=15, y=0.985); fig.text(0.5, 0.915, l1 + "\n" + l2, ha="center", va="top", fontsize=11)
        fig.text(0.01, 0.008, FOOT, fontsize=8, va="bottom")
        b = io.BytesIO(); fig.savefig(b, format="png", dpi=300, bbox_inches="tight"); plt.close(fig); out[case] = b.getvalue()
        print("case %s: %s | %s" % (case, l1, l2))
    for c, b in out.items():
        p = ss.check_path(os.path.join(R, F + NM[c]))
        with open(p, "xb") as f: f.write(b)
        assert hashlib.sha256(open(p, "rb").read()).hexdigest() == hashlib.sha256(b).hexdigest(); print("saved %-26s %7d bytes sha12 %s" % (NM[c], len(b), hashlib.sha256(b).hexdigest()[:12]))
_c184()

In [ ]:
# Cell 185 (08_story_extras.ipynb): v4 worked-example figures (B+C pair, A supplement). Reads saved tables; exclusive create. No restart.
def _c185():
    import os, sys, io, hashlib, textwrap, warnings
    import numpy as np, pandas as pd
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    from matplotlib.ticker import FuncFormatter
    warnings.filterwarnings("ignore")
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sys.path.insert(0, R + "/shared/src"); import safe_save as ss
    F = "selection/figures/"; NM = {"pair": "sel_story_pair_BC_v4.png", "A": "sel_story_case_A_supplement_v4.png"}
    for n in NM.values(): ss.check_path(os.path.join(R, F + n))
    T = 1 / 11; INK, FLIP, GREY = "#222222", "#D55E00", "#B0B0B0"; XMAX = 0.40
    ex = pd.read_csv(R + "/selection/tables/sel_worked_examples.csv")
    NICE = {"Age": "Age", "Debt Ratio": "Debt ratio", "Credit Utilisation Rate": "Credit utilisation", "30-59 Days Past Due": "30–59 days past due", "60-89 Days Past Due": "60–89 days past due", "90+ Days Late": "90+ days past due"}
    TITLE = {"A": "Case A: flagged at both stages", "B": "Case B: not flagged at application, flagged at monitoring", "C": "Case C: flagged at both stages, close to the line"}
    FOOT = ("Sensitivity to unseen evidence, not the reason for a flag. Flag threshold 1/11 assumes a missed defaulter costs 10x a wrongly flagged borrower (r=10, assumed); at r=5 the application-stage model flags nobody. "
            "Cases chosen by rules fixed before outcomes were seen; illustrations, not typical borrowers. Fairness not assessed.")
    CAP = ("Each open circle removes ONE fact at a time. A flip shows the borrower is close to the line, not why they were flagged. Rows are sorted by the shown shift. "
           "Application and monitoring risks come from two different models (selection BN, 6 inputs; full BN, 10 inputs) answering different questions, so they need not agree. "
           "Facts not shown have exactly zero effect in this model (outside Default's Markov blanket in the learned structure). "
           "Overall mean predicted risk matches the observed rate (6.69% vs 6.68%); application-stage cell values are training default rates. "
           "'Unknown' means the model averages over the possible values, weighted by how likely each is given the rest of the evidence, not set to zero.")
    pct = FuncFormatter(lambda v, _: "%d%%" % round(v * 100))
    def rows_for(case):
        d = ex[ex.case == case].copy(); d["shown"] = (d.p_retracted - d.stage_baseline).abs()
        out = []
        for stage, lab in (("selection", "Application stage"), ("monitoring", "Monitoring stage")):
            s = d[d.stage == stage].sort_values("shown", ascending=False); out.append((lab, stage, float(s.stage_baseline.iloc[0]), s))
        return out
    def draw(ax, case, annotate_first=False):
        blocks = rows_for(case); y = 0; ylabels = []; yticks = []
        total = sum(len(b[3]) + 1 for b in blocks); y = total
        ax.axvspan(T, XMAX, color="#F0F0F0", zorder=0); ax.axvline(T, color="black", ls="--", lw=1.3, zorder=1)
        ax.text(T + 0.006, total + 0.55, "flagged region", fontsize=12, color="dimgray", va="bottom"); ax.text(T - 0.006, total + 0.55, "not flagged", fontsize=12, color="dimgray", va="bottom", ha="right")
        first = True
        for lab, stage, base, s in blocks:
            ax.text(0.004, y, "%s: risk %.1f%% (%s)" % (lab, 100 * base, "flagged" if base >= T else "not flagged"), fontsize=14, fontweight="bold", va="center", color=INK); y -= 1
            for r in s.itertuples():
                flip = bool(r.realised_flip); c = FLIP if flip else GREY; yticks.append(y); ylabels.append((NICE[r.variable], flip))
                ax.plot([base, r.p_retracted], [y, y], color=c, lw=4.0 if flip else 1.8, zorder=2, solid_capstyle="butt")
                ax.scatter([base], [y], s=170, color=INK, zorder=3)
                ax.scatter([r.p_retracted], [y], s=210, facecolor="white", edgecolor=FLIP if flip else "#707070", linewidth=2.8, zorder=5)
                txt = "≈ same" if r.shown < 0.005 else "%.1f%%" % (100 * r.p_retracted)
                ax.text(r.p_retracted, y + 0.36, txt, ha="center", va="bottom", fontsize=12, color="#444444", zorder=6, bbox=dict(fc="white", ec="none", pad=1.2))
                if flip: ax.text(max(base, r.p_retracted) + 0.014, y - 0.02, "FLAG FLIPS IF UNKNOWN", va="center", fontsize=12.5, fontweight="bold", color=FLIP, zorder=6, bbox=dict(fc="white", ec="none", pad=1.0))
                if annotate_first and first:
                    ax.annotate("with all facts known", (base, y), (base + 0.02, y + 0.62), fontsize=11, color=INK, arrowprops=dict(arrowstyle="-", color="gray", lw=0.8), zorder=7)
                    ax.annotate("if this fact were unknown", (r.p_retracted, y), (max(0.0, r.p_retracted - 0.06), y - 0.62), fontsize=11, color=INK, arrowprops=dict(arrowstyle="-", color="gray", lw=0.8), zorder=7)
                first = False; y -= 1
        ax.set_yticks(yticks); ax.set_yticklabels([l for l, _ in ylabels], fontsize=14)
        for tl, (_, fl) in zip(ax.get_yticklabels(), ylabels):
            if fl: tl.set_color(FLIP); tl.set_fontweight("bold")
        ax.set_xlim(0, XMAX); ax.set_ylim(0.2, total + 1.3); ax.xaxis.set_major_formatter(pct); ax.tick_params(axis="x", labelsize=13); ax.tick_params(axis="y", length=0)
        ax.set_xlabel("Estimated probability of default", fontsize=14); ax.grid(False)
        for sp in ("top", "right", "left"): ax.spines[sp].set_visible(False)
        ax.set_title(TITLE[case], fontsize=16, pad=14, loc="left")
    legend = [Line2D([0], [0], marker="o", color="w", markerfacecolor=INK, markersize=12, label="filled dot: risk with all facts known"),
              Line2D([0], [0], marker="o", color="w", markerfacecolor="white", markeredgecolor="#707070", markeredgewidth=2.5, markersize=12, label="open circle: risk if that one fact were unknown"),
              Line2D([0], [0], color=FLIP, lw=4, label="vermilion: the flag would flip"), Line2D([0], [0], color=GREY, lw=2, label="grey: the flag would not flip")]
    def finish(fig, width):
        fig.legend(handles=legend, loc="lower center", ncol=2, frameon=False, fontsize=13, bbox_to_anchor=(0.5, 0.215))
        fig.text(0.02, 0.19, textwrap.fill(CAP, int(width * 8.6)), fontsize=12, va="top"); fig.text(0.02, 0.095, textwrap.fill(FOOT, int(width * 8.6)), fontsize=11, va="top", style="italic")
        b = io.BytesIO(); fig.savefig(b, format="png", dpi=300, bbox_inches="tight"); plt.close(fig); return b.getvalue()
    out = {}
    fig, axes = plt.subplots(1, 2, figsize=(20, 10.5)); fig.subplots_adjust(left=0.11, right=0.985, top=0.9, bottom=0.37, wspace=0.62)
    draw(axes[0], "B", annotate_first=True); draw(axes[1], "C"); out["pair"] = finish(fig, 20)
    fig, ax = plt.subplots(figsize=(11, 10.5)); fig.subplots_adjust(left=0.2, right=0.97, top=0.9, bottom=0.37)
    draw(ax, "A", annotate_first=True); out["A"] = finish(fig, 11)
    print("caption:", CAP, "\n\nfooter:", FOOT)
    for k, b in out.items():
        p = ss.check_path(os.path.join(R, F + NM[k]))
        with open(p, "xb") as f: f.write(b)
        assert hashlib.sha256(open(p, "rb").read()).hexdigest() == hashlib.sha256(b).hexdigest(); print("saved %-38s %7d bytes sha12 %s" % (NM[k], len(b), hashlib.sha256(b).hexdigest()[:12]))
_c185()

In [ ]:
# Cell 186 (08_story_extras.ipynb): v5 worked-example figures (chart + legend + footer only) + caption txt. Exclusive create. No restart.
def _c186():
    import os, sys, io, hashlib, textwrap, warnings
    import numpy as np, pandas as pd
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    from matplotlib.ticker import FuncFormatter
    warnings.filterwarnings("ignore")
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sys.path.insert(0, R + "/shared/src"); import safe_save as ss
    F = "selection/figures/"; NM = {"pair": F + "sel_story_pair_BC_v5.png", "A": F + "sel_story_case_A_supplement_v5.png", "cap": "selection/tables/sel_story_captions.txt"}
    for n in NM.values(): ss.check_path(os.path.join(R, n))
    T = 1 / 11; INK, FLIP, GREY = "#222222", "#D55E00", "#B0B0B0"; XMAX = 0.40
    ex = pd.read_csv(R + "/selection/tables/sel_worked_examples.csv")
    NICE = {"Age": "Age", "Debt Ratio": "Debt ratio", "Credit Utilisation Rate": "Credit utilisation", "30-59 Days Past Due": "30–59 days past due", "60-89 Days Past Due": "60–89 days past due", "90+ Days Late": "90+ days past due"}
    TITLE = {"A": "Case A: flagged at both stages", "B": "Case B: not flagged at application, flagged at monitoring", "C": "Case C: flagged at both stages, close to the line"}
    FOOT = ("Sensitivity to unseen evidence, not the reason for a flag. Flag threshold 1/11 assumes a missed defaulter costs 10x a wrongly flagged borrower (r=10, assumed); at r=5 the application-stage model flags nobody. "
            "Cases chosen by rules fixed before outcomes were seen; illustrations, not typical borrowers. Fairness not assessed.")
    CAP = ("Each open circle removes ONE fact at a time. A flip shows the borrower is close to the line, not why they were flagged. Rows are sorted by the shown shift (sorting by expected shift S_exp is in sel_worked_examples.csv).\n\n"
           "Application and monitoring risks come from two different models (selection BN, 6 inputs; full BN, 10 inputs) answering different questions, so they need not agree.\n\n"
           "Facts not shown have exactly zero effect in this model (outside Default's Markov blanket in the learned structure).\n\n"
           "Overall mean predicted risk matches the observed rate (6.69% vs 6.68%); application-stage cell values are training default rates.\n\n"
           "'Unknown' means the model averages over the possible values, weighted by how likely each is given the rest of the evidence, not set to zero.\n\n"
           "Footer (printed on the figure): " + FOOT + "\n")
    pct = FuncFormatter(lambda v, _: "%d%%" % round(v * 100))
    def draw(ax, case):
        d = ex[ex.case == case].copy(); d["shown"] = (d.p_retracted - d.stage_baseline).abs(); y = 0.0; ticks, labs = [], []
        ax.axvspan(T, XMAX, color="#F0F0F0", zorder=0); ax.axvline(T, color="black", ls="--", lw=1.3, zorder=1)
        blocks = []
        for stage, lab in (("selection", "Application stage"), ("monitoring", "Monitoring stage")):
            s = d[d.stage == stage].sort_values("shown", ascending=False); blocks.append((lab, float(s.stage_baseline.iloc[0]), s))
        n = sum(len(b[2]) + 2 for b in blocks); y = n
        ax.text(T - 0.006, n + 0.6, "not flagged", ha="right", fontsize=12, color="dimgray"); ax.text(T + 0.006, n + 0.6, "flagged", ha="left", fontsize=12, color="dimgray")
        for lab, base, s in blocks:
            y -= 0.6
            ax.text(-0.355, y, "%s: risk %.1f%% (%s)" % (lab, 100 * base, "flagged" if base >= T else "not flagged"), transform=ax.transData, fontsize=14, fontweight="bold", va="center", ha="left", color=INK, clip_on=False); y -= 1.2
            for r in s.itertuples():
                flip = bool(r.realised_flip); c = FLIP if flip else GREY; ticks.append(y); labs.append((NICE[r.variable], flip))
                ax.plot([base, r.p_retracted], [y, y], color=c, lw=4.2 if flip else 1.8, zorder=2, solid_capstyle="butt")
                ax.scatter([base], [y], s=170, color=INK, zorder=3)
                ax.scatter([r.p_retracted], [y], s=230, facecolor="white", edgecolor=FLIP if flip else "#707070", linewidth=2.8, zorder=5)
                if r.shown >= 0.005:
                    ax.text(r.p_retracted + (0.016 if r.p_retracted >= base else -0.016), y, "%.1f%%" % (100 * r.p_retracted), ha="left" if r.p_retracted >= base else "right", va="center", fontsize=12.5, color="#444444", zorder=6)
                else:
                    ax.text(max(base, r.p_retracted) + 0.02, y, "no visible change", ha="left", va="center", fontsize=12, color="#444444", zorder=6)
                if flip: ax.text(0.405, y, "FLAG FLIPS IF UNKNOWN", va="center", ha="left", fontsize=12.5, fontweight="bold", color=FLIP, clip_on=False)
                y -= 1
        ax.set_yticks(ticks); ax.set_yticklabels([l for l, _ in labs], fontsize=14)
        for tl, (_, fl) in zip(ax.get_yticklabels(), labs):
            if fl: tl.set_color(FLIP); tl.set_fontweight("bold")
        ax.set_xlim(0, XMAX); ax.set_ylim(y + 0.2, n + 1.3); ax.xaxis.set_major_formatter(pct); ax.tick_params(axis="x", labelsize=13); ax.tick_params(axis="y", length=0)
        ax.set_xlabel("Estimated probability of default", fontsize=14); ax.grid(False)
        for sp in ("top", "right", "left"): ax.spines[sp].set_visible(False)
        ax.set_title(TITLE[case], fontsize=16, pad=16, loc="left")
    legend = [Line2D([0], [0], marker="o", color="w", markerfacecolor=INK, markersize=12, label="filled dot: risk with all facts known"),
              Line2D([0], [0], marker="o", color="w", markerfacecolor="white", markeredgecolor="#707070", markeredgewidth=2.5, markersize=12, label="open circle: risk if that one fact were unknown"),
              Line2D([0], [0], color=FLIP, lw=4, label="vermilion: the flag would flip"), Line2D([0], [0], color=GREY, lw=2, label="grey: the flag would not flip")]
    def finish(fig, wrap):
        fig.legend(handles=legend, loc="lower center", ncol=4, frameon=False, fontsize=12.5, bbox_to_anchor=(0.5, 0.075))
        fig.text(0.02, 0.045, textwrap.fill(FOOT, wrap), fontsize=11, va="top", style="italic", linespacing=1.5)
        b = io.BytesIO(); fig.savefig(b, format="png", dpi=300, bbox_inches="tight"); plt.close(fig); return b.getvalue()
    out = {}
    fig, axes = plt.subplots(1, 2, figsize=(22, 9.6)); fig.subplots_adjust(left=0.12, right=0.9, top=0.9, bottom=0.26, wspace=0.95)
    draw(axes[0], "B"); draw(axes[1], "C"); out["pair"] = finish(fig, 215)
    fig, ax = plt.subplots(figsize=(13, 9.6)); fig.subplots_adjust(left=0.2, right=0.78, top=0.9, bottom=0.26)
    draw(ax, "A"); out["A"] = finish(fig, 125)
    for k, b in out.items():
        p = ss.check_path(os.path.join(R, NM[k]))
        with open(p, "xb") as f: f.write(b)
        print("saved %-60s %7d bytes sha12 %s" % (NM[k], len(b), hashlib.sha256(b).hexdigest()[:12]))
    p = ss.check_path(os.path.join(R, NM["cap"]))
    with open(p, "x", encoding="utf-8") as f: f.write(CAP)
    print("saved %s" % NM["cap"])
_c186()

In [ ]:
# Cell 187 (08_story_extras.ipynb): v6 worked-example figures. Exclusive create. No restart.
def _c187():
    import os, sys, io, hashlib, textwrap, warnings
    import numpy as np, pandas as pd
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    from matplotlib.ticker import FuncFormatter
    warnings.filterwarnings("ignore")
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sys.path.insert(0, R + "/shared/src"); import safe_save as ss
    F = "selection/figures/"; NM = {"pair": F + "sel_story_pair_BC_v6.png", "A": F + "sel_story_case_A_supplement_v6.png"}
    for n in NM.values(): ss.check_path(os.path.join(R, n))
    T = 1 / 11; INK, FLIP, GREY = "#222222", "#D55E00", "#B0B0B0"; XMAX = 0.40
    ex = pd.read_csv(R + "/selection/tables/sel_worked_examples.csv")
    NICE = {"Age": "Age", "Debt Ratio": "Debt ratio", "Credit Utilisation Rate": "Credit utilisation", "30-59 Days Past Due": "30–59 days past due", "60-89 Days Past Due": "60–89 days past due", "90+ Days Late": "90+ days past due"}
    TITLE = {"A": "Case A: flagged at both stages", "B": "Case B: not flagged at application, flagged at monitoring", "C": "Case C: flagged at both stages, close to the line"}
    FOOT = ("Sensitivity to unseen evidence, not the reason for a flag. Flag threshold 1/11 assumes a missed defaulter costs 10x a wrongly flagged borrower (r=10, assumed); at r=5 the application-stage model flags nobody. "
            "Cases chosen by rules fixed before outcomes were seen; illustrations, not typical borrowers. Fairness not assessed.")
    pct = FuncFormatter(lambda v, _: "%d%%" % round(v * 100)); WB = dict(fc="white", ec="none", pad=1.5)
    def draw(ax, case):
        d = ex[ex.case == case].copy(); d["shown"] = (d.p_retracted - d.stage_baseline).abs(); ticks, labs = [], []
        ax.axvspan(T, XMAX, color="#F0F0F0", zorder=0); ax.axvline(T, color="black", ls="--", lw=1.3, zorder=1)
        blocks = []
        for stage, lab in (("selection", "Application stage"), ("monitoring", "Monitoring stage")):
            s = d[d.stage == stage].sort_values("shown", ascending=False); blocks.append((lab, float(s.stage_baseline.iloc[0]), s))
        n = sum(len(b[2]) + 2 for b in blocks); y = n
        ax.text(T - 0.006, n + 0.6, "not flagged", ha="right", fontsize=12, color="dimgray"); ax.text(T + 0.006, n + 0.6, "flagged", ha="left", fontsize=12, color="dimgray")
        for lab, base, s in blocks:
            y -= 0.6
            ax.text(0.0, y, "%s: risk %.1f%% (%s)" % (lab, 100 * base, "flagged" if base >= T else "not flagged"), fontsize=14, fontweight="bold", va="center", ha="left", color=INK, zorder=7, bbox=WB); y -= 1.2
            for r in s.itertuples():
                flip = bool(r.realised_flip); c = FLIP if flip else GREY; ticks.append(y); labs.append((NICE[r.variable], flip))
                ax.plot([base, r.p_retracted], [y, y], color=c, lw=4.2 if flip else 1.8, zorder=2, solid_capstyle="butt")
                ax.scatter([r.p_retracted], [y], s=230, facecolor="white", edgecolor=FLIP if flip else "#707070", linewidth=2.8, zorder=4)
                ax.scatter([base], [y], s=120 if r.shown < 0.02 else 170, color=INK, zorder=5, edgecolor="white", linewidth=1.2)
                if r.shown < 0.02:
                    ax.text(max(base, r.p_retracted) + 0.02, y, "no visible change", ha="left", va="center", fontsize=12, color="#444444", zorder=6, bbox=WB)
                else:
                    right = r.p_retracted >= base; x = r.p_retracted + (0.016 if right else -0.016)
                    if (not right) and abs(x - T) < 0.03: x = r.p_retracted - 0.016
                    ax.text(x, y, "%.1f%%" % (100 * r.p_retracted), ha="left" if right else "right", va="center", fontsize=12.5, color="#444444", zorder=6, bbox=WB)
                if flip: ax.text(XMAX + 0.01, y, "FLAG FLIPS IF UNKNOWN", va="center", ha="left", fontsize=12.5, fontweight="bold", color=FLIP, clip_on=False)
                y -= 1
        ax.set_yticks(ticks); ax.set_yticklabels([l for l, _ in labs], fontsize=14)
        for tl, (_, fl) in zip(ax.get_yticklabels(), labs):
            if fl: tl.set_color(FLIP); tl.set_fontweight("bold")
        ax.set_xlim(0, XMAX); ax.set_ylim(y + 0.2, n + 1.3); ax.xaxis.set_major_formatter(pct); ax.tick_params(axis="x", labelsize=13); ax.tick_params(axis="y", length=0, pad=8)
        ax.set_xlabel("Estimated probability of default", fontsize=14); ax.grid(False)
        for sp in ("top", "right", "left"): ax.spines[sp].set_visible(False)
        ax.set_title(TITLE[case], fontsize=16, pad=16, loc="left")
    legend = [Line2D([0], [0], marker="o", color="w", markerfacecolor=INK, markersize=12, label="filled dot: risk with all facts known"),
              Line2D([0], [0], marker="o", color="w", markerfacecolor="white", markeredgecolor="#707070", markeredgewidth=2.5, markersize=12, label="open circle: risk if that one fact were unknown"),
              Line2D([0], [0], color=FLIP, lw=4, label="vermilion: the flag would flip"), Line2D([0], [0], color=GREY, lw=2, label="grey: the flag would not flip")]
    def finish(fig, wrap, ncol):
        fig.legend(handles=legend, loc="lower center", ncol=ncol, frameon=False, fontsize=12.5, bbox_to_anchor=(0.5, 0.06 if ncol == 4 else 0.035))
        fig.text(0.02, 0.0, textwrap.fill(FOOT, wrap), fontsize=11, va="top", style="italic", linespacing=1.5)
        b = io.BytesIO(); fig.savefig(b, format="png", dpi=300, bbox_inches="tight"); plt.close(fig); return b.getvalue()
    out = {}
    fig, axes = plt.subplots(1, 2, figsize=(22, 9.6)); fig.subplots_adjust(left=0.12, right=0.9, top=0.9, bottom=0.26, wspace=0.95)
    draw(axes[0], "B"); draw(axes[1], "C"); out["pair"] = finish(fig, 215, 4)
    fig, ax = plt.subplots(figsize=(12, 10)); fig.subplots_adjust(left=0.24, right=0.76, top=0.9, bottom=0.3)
    draw(ax, "A"); out["A"] = finish(fig, 110, 2)
    for k, b in out.items():
        p = ss.check_path(os.path.join(R, NM[k]))
        with open(p, "xb") as f: f.write(b)
        print("saved %-60s %7d bytes sha12 %s" % (NM[k], len(b), hashlib.sha256(b).hexdigest()[:12]))
_c187()

In [ ]:
# Cell 188 (08_story_extras.ipynb): Phase 7b manifest/git DRY RUN. Writes NOTHING. No restart.
def _c188():
    import os, csv, time, hashlib, subprocess
    R = os.environ["CREDIT_BN_ROOT"]
    TAG12 = "5626192403ef"
    git = lambda *a: subprocess.run(["git", "-C", R, *a], capture_output=True, text=True).stdout.strip()
    def sha(p):
        h = hashlib.sha256()
        with open(p, 'rb') as f:
            for b in iter(lambda: f.read(1 << 20), b''):
                h.update(b)
        return h.hexdigest()
    mp = os.path.join(R, "MANIFEST.csv"); raw = open(mp, 'rb').read()
    existing = {r['path'] for r in csv.DictReader(open(mp, newline=''))}
    print("DRY_RUN = True")
    print("manifest rows now:", len(existing), "(expected 207) | ends with newline:", raw.endswith(b'\n'))
    tag = git("rev-list", "-n", "1", "full-variable-final"); print("tag points to:", tag[:12], "| OK:", tag.startswith(TAG12))
    print("git status before:", git("status", "--porcelain").splitlines())
    rows = []
    for top in ("selection", "shared", "monitoring_rq3"):
        for dp, dn, fn in os.walk(os.path.join(R, top)):
            dn[:] = [d for d in dn if d not in (".ipynb_checkpoints", "__pycache__")]
            for f in sorted(fn):
                if f.endswith(".pyc") or f == ".DS_Store": continue
                p = os.path.join(dp, f); rel = os.path.relpath(p, R)
                if rel not in existing: rows.append((rel, os.path.getsize(p), sha(p)[:12]))
    print("\nrows to append (%d):" % len(rows))
    for r in sorted(rows): print("  ", *r)
    print("expected manifest rows after:", len(existing) + len(rows))
    print("\nwould stage ONLY: MANIFEST.csv, shared/src/decision_value.py, selection/notebooks/08_story_extras.ipynb (not 03_-07_ notebooks, Reminder.md, Writing_Edits.md)")
    print("DRY RUN ONLY: nothing written.")
_c188()

In [ ]:
# Cell 188b: why is the manifest 205 rows? READ-ONLY, writes NOTHING.
def _c188b():
    import csv, io, subprocess
    import os; R = os.environ["CREDIT_BN_ROOT"]
    git = lambda *a: subprocess.run(["git", "-C", R, *a], capture_output=True, text=True).stdout
    def paths(txt): return [r["path"] for r in csv.DictReader(io.StringIO(txt))]
    disk = paths(open(R + "/MANIFEST.csv").read())
    print("rows on disk:", len(disk), "| duplicates:", len(disk) - len(set(disk)))
    for c in git("log", "--format=%h %s", "-6", "--", "MANIFEST.csv").splitlines():
        h = c.split()[0]; p = paths(git("show", h + ":MANIFEST.csv")); print("%-8s rows %3d | %s" % (h, len(p), c[8:70]))
    head = paths(git("show", "HEAD:MANIFEST.csv")); print("HEAD rows:", len(head))
    print("in HEAD but missing on disk:", sorted(set(head) - set(disk)))
    print("on disk but not in HEAD:", sorted(set(disk) - set(head)))
    print("git diff stat for MANIFEST.csv:", git("diff", "--stat", "--", "MANIFEST.csv").strip() or "no change vs HEAD")
_c188b()

In [ ]:
# Cell 189 (08_story_extras.ipynb): REAL RUN (approved). Manifest append (31 rows) + NEW git commit. Tag never touched.
def _c189():
    import os, csv, time, hashlib, subprocess
    R = os.environ["CREDIT_BN_ROOT"]
    TAG12 = "5626192403ef"; NOTE = "added after Phase 7b, 2026-10-17"
    NB = "selection/notebooks/08_story_extras.ipynb"; DV = "shared/src/decision_value.py"
    def git(*a, check=True):
        r = subprocess.run(["git", "-C", R, *a], capture_output=True, text=True)
        if check and r.returncode != 0: raise RuntimeError("git %s failed: %s" % (" ".join(a), r.stderr.strip()[:300]))
        return r.stdout.strip()
    def sha(p):
        h = hashlib.sha256()
        with open(p, 'rb') as f:
            for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
        return h.hexdigest()
    mp = os.path.join(R, "MANIFEST.csv"); raw0 = open(mp, 'rb').read()
    recs = list(csv.DictReader(open(mp, newline=''))); existing = {r['path'] for r in recs}
    assert raw0.endswith(b'\n') and len(recs) == 207, "unexpected manifest state: STOP"
    assert git("rev-list", "-n", "1", "full-variable-final").startswith(TAG12), "tag moved: STOP"
    SUP = "; SUPERSEDED, kept on disk"; CAND = "; candidate: user to confirm v3 vs latest (user prefers v3 style)"
    F = [("monitoring_rq3/mon_decision_value_monitoring.csv", "1490ad910719", ""), ("monitoring_rq3/mon_oof_predictions.pkl", "d04139c187c8", ""),
         ("selection/figures/sel_decision_value_combined.png", "bd970301a4ef", ""), ("selection/figures/sel_stage_flow.png", "ce10e27c460c", SUP),
         ("selection/figures/sel_stage_flow_v2.png", "ff62813627d6", SUP), ("selection/figures/sel_story_case_A_supplement_v4.png", "4a8d3627f1f8", SUP),
         ("selection/figures/sel_story_case_A_supplement_v5.png", "3c31862e1d7c", SUP), ("selection/figures/sel_story_case_A_supplement_v6.png", "9dedb2968eaf", CAND),
         ("selection/figures/sel_story_case_A_v2.png", "b5de87d1e663", SUP), ("selection/figures/sel_story_case_A_v3.png", "01c0d5954eb2", CAND),
         ("selection/figures/sel_story_case_B_v2.png", "35340da2630b", SUP), ("selection/figures/sel_story_case_B_v3.png", "ee0c37b86dde", CAND),
         ("selection/figures/sel_story_case_C_v2.png", "eb664159aaab", SUP), ("selection/figures/sel_story_case_C_v3.png", "70612e204a74", CAND),
         ("selection/figures/sel_story_pair_BC_v4.png", "9e6ac6259c7b", SUP), ("selection/figures/sel_story_pair_BC_v5.png", "6de30bd53642", SUP),
         ("selection/figures/sel_story_pair_BC_v6.png", "10362474bf53", CAND + "; known fault: Case C 60-89 row label clipped by value label"),
         ("selection/figures/sel_worked_age_bubbles.png", "888b66c68aee", SUP), ("selection/figures/sel_worked_fan.png", "84b297dd32d5", SUP),
         ("selection/figures/sel_worked_journey.png", "21ad935e6bef", SUP), (NB, None, ""),
         ("selection/tables/sel_decision_value_combined.csv", "57776a7d8010", ""), ("selection/tables/sel_decision_value_selection.csv", "2b290f28f02a", ""),
         ("selection/tables/sel_info_ladder_bootstrap.csv", "967226d30bd9", ""), ("selection/tables/sel_info_ladder_table.csv", "c8781fc04f89", ""),
         ("selection/tables/sel_stage_agreement.csv", "828b4454f0a9", ""), ("selection/tables/sel_story_captions.txt", "3a62fe4e8a66", ""),
         ("selection/tables/sel_worked_example_completions.csv", "bf84ff6989b2", ""), ("selection/tables/sel_worked_examples.csv", "f646fec3d9b8", ""),
         ("selection/tables/sel_worked_examples_notes.json", "f12d508b0a63", ""), (DV, "c1476cef61d1", "")]
    rows = []
    for rel, h12, extra in F:
        p = os.path.join(R, rel); assert rel not in existing and os.path.exists(p), "unexpected state for %s: STOP" % rel
        h = sha(p)
        if h12: assert h.startswith(h12), "hash changed since dry run for %s: STOP" % rel
        else: print("notebook hash recorded now: %s (dry run showed 96d957a79a36)" % h[:12])
        role = "shared" if rel.startswith("shared/") else ("monitoring_rq3" if rel.startswith("monitoring_rq3") else "selection")
        rows.append([rel, os.path.getsize(p), h, role, time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(p))), NOTE + extra])
    with open(mp, 'a', newline='') as fh: csv.writer(fh).writerows(rows)
    n = len(list(csv.DictReader(open(mp, newline=''))))
    assert n == 238 and open(mp, 'rb').read().startswith(raw0), "manifest check failed: STOP (not committed)"
    git("add", "MANIFEST.csv", DV, NB)
    ident = [] if (git("config", "user.name", check=False) and git("config", "user.email", check=False)) else ["-c", "user.name=Ahmed Nanabhay", "-c", "user.email=noreply@local.invalid"]
    git(*ident, "commit", "-q", "-m", "Phase 7b: decision value, information ladder, stage agreement, worked examples, figures")
    print("manifest rows now:", n, "(expected 238)"); print("new commit:", git("log", "--oneline", "-1"))
    print("tag unchanged:", git("rev-list", "-n", "1", "full-variable-final").startswith(TAG12)); print("git status:", git("status", "--porcelain").splitlines())
_c189()

In [ ]:
# Cell 190 (08_story_extras.ipynb): v7 story cards (pair BC, supplement A), stage flow v3, captions v2, collision + greyscale checks. Exclusive create. No restart.
def _c190():
    import os, sys, io, hashlib, textwrap, warnings
    import numpy as np, pandas as pd
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    from matplotlib.patches import Rectangle
    from matplotlib.ticker import FuncFormatter
    from PIL import Image
    warnings.filterwarnings("ignore")
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sys.path.insert(0, R + "/shared/src"); import safe_save as ss
    F = "selection/figures/"
    NM = {"pair": F + "sel_story_pair_BC_v7.png", "A": F + "sel_story_case_A_supplement_v7.png", "flow": F + "sel_stage_flow_v3.png", "cap": "selection/tables/sel_story_captions_v2.txt"}
    for n in NM.values(): ss.check_path(os.path.join(R, n))
    T = 1 / 11; BLUE, GREEN, FLIP, GREY = "#0072B2", "tab:green", "#D55E00", "#9a9a9a"; XMAX = 0.40
    ex = pd.read_csv(R + "/selection/tables/sel_worked_examples.csv"); ag = pd.read_csv(R + "/selection/tables/sel_stage_agreement.csv")
    NICE = {"Age": "Age", "Debt Ratio": "Debt ratio", "Credit Utilisation Rate": "Credit utilisation", "30-59 Days Past Due": "30–59 days past due", "60-89 Days Past Due": "60–89 days past due", "90+ Days Late": "90+ days past due"}
    TITLE = {"A": "Case A: flagged at both stages", "B": "Case B: not flagged at application, flagged at monitoring", "C": "Case C: flagged at both stages, close to the line"}
    FOOT = ("Sensitivity to unseen evidence, not the reason for a flag. Flag threshold 1/11 assumes a missed defaulter costs 10x a wrongly flagged borrower (r=10, assumed); at r=5 the application-stage model flags nobody. "
            "Cases chosen by rules fixed before outcomes were seen; illustrations, not typical borrowers. Fairness not assessed.")
    FOOT_FLOW = ("Flag threshold 1/11 assumes a missed defaulter costs 10x a wrongly flagged borrower (r=10, assumed); at r=5 the application-stage model flags nobody. "
                 "Application and monitoring flags come from two different models; flags are not decisions.")
    CAP = ("Each open marker removes ONE fact at a time. Rows are sorted by the shown shift |risk without the fact - risk with all facts| (the expected-shift S_exp order is in sel_worked_examples.csv).\n\n"
           "(i) Application and monitoring risks come from two different models (selection BN, 6 inputs; full BN, 10 inputs) answering different questions, so they need not agree.\n\n"
           "(ii) Facts not shown have exactly zero effect in this model (outside Default's Markov blanket in the learned structure).\n\n"
           "(iii) Overall mean predicted risk matches the observed rate (6.69% vs 6.68%); application-stage cell values are training default rates.\n\n"
           "(iv) 'Unknown' means the model averages over the possible values, weighted by how likely each is given the rest of the evidence, not set to zero.\n\n"
           "(v) Age is a direct input of the application-stage model; fairness not assessed; a flip shows the borrower is close to the line, not why they were flagged.\n\n"
           "Footer (printed on every worked-example figure): " + FOOT + "\n")
    pct = FuncFormatter(lambda v, _: "%d%%" % round(v * 100)); WB = dict(fc="white", ec="none", pad=1.5)
    problems = []
    def draw(ax, case):
        d = ex[ex.case == case].copy(); d["shown"] = (d.p_retracted - d.stage_baseline).abs(); ticks, labs, texts = [], [], []
        ax.axvspan(T, XMAX, color="#F3F3F3", zorder=0); ax.axvline(T, color="black", ls="--", lw=1.3, zorder=1)
        blocks = []
        for stage, lab, col in (("selection", "Application stage", BLUE), ("monitoring", "Monitoring stage", GREEN)):
            s = d[d.stage == stage].sort_values("shown", ascending=False); blocks.append((lab, col, float(s.stage_baseline.iloc[0]), s))
        n = sum(len(b[3]) + 2 for b in blocks); y = n
        ax.text(T - 0.006, n + 0.7, "not flagged", ha="right", fontsize=12, color="dimgray"); ax.text(T + 0.006, n + 0.7, "flagged", ha="left", fontsize=12, color="dimgray")
        for lab, col, base, s in blocks:
            y -= 0.6; ax.text(0.005, y + 0.62, "%s: risk %.1f%% (%s)" % (lab, 100 * base, "flagged" if base >= T else "not flagged"), fontsize=14, fontweight="bold", va="center", ha="left", color=col); y -= 1.3
            for r in s.itertuples():
                flip = bool(r.realised_flip); c = FLIP if flip else GREY; ticks.append(y); labs.append((NICE[r.variable], flip))
                ax.plot([base, r.p_retracted], [y, y], color=c, lw=4.0 if flip else 1.8, zorder=2, solid_capstyle="butt")
                ax.scatter([r.p_retracted], [y], s=250 if flip else 210, marker="D" if flip else "o", facecolor="white", edgecolor=FLIP if flip else "#707070", linewidth=2.8, zorder=4)
                ax.scatter([base], [y], s=150, color=col, zorder=5, edgecolor="white", linewidth=1.2)
                t = ax.text(r.p_retracted, y + 0.42, ("no visible change" if r.shown < 0.02 else "%.1f%%" % (100 * r.p_retracted)), ha="center", va="bottom", fontsize=12, color="#333333", zorder=6, bbox=WB)
                texts.append((t, r.p_retracted))
                if flip: ax.text(XMAX + 0.012, y, "FLAG FLIPS IF UNKNOWN", va="center", ha="left", fontsize=12.5, fontweight="bold", color=FLIP, clip_on=False)
                y -= 1
        ax.set_yticks(ticks); ax.set_yticklabels([l for l, _ in labs], fontsize=14)
        for tl, (_, fl) in zip(ax.get_yticklabels(), labs):
            if fl: tl.set_color(FLIP); tl.set_fontweight("bold")
        ax.set_xlim(0, XMAX); ax.set_ylim(y + 0.2, n + 1.4); ax.xaxis.set_major_formatter(pct); ax.tick_params(axis="x", labelsize=13); ax.tick_params(axis="y", length=0, pad=8)
        ax.set_xlabel("Estimated probability of default", fontsize=14); ax.grid(False)
        for sp in ("top", "right", "left"): ax.spines[sp].set_visible(False)
        ax.set_title(TITLE[case], fontsize=16, pad=16, loc="left"); return texts
    legend = [Line2D([0], [0], marker="o", color="w", markerfacecolor=BLUE, markersize=12, label="blue / green dot: risk with all facts known (application / monitoring)"),
              Line2D([0], [0], marker="o", color="w", markerfacecolor="white", markeredgecolor="#707070", markeredgewidth=2.5, markersize=12, label="open circle: risk if that one fact were unknown"),
              Line2D([0], [0], marker="D", color=FLIP, markerfacecolor="white", markeredgewidth=2.5, markersize=11, lw=3, label="open diamond + orange line + tag: the flag would flip")]
    def check_overlaps(fig, ax, texts, tag):
        fig.canvas.draw(); rd = fig.canvas.get_renderer(); line_x = ax.transData.transform((T, 0))[0]
        boxes = [(t, t.get_window_extent(rd)) for t, _ in texts]
        rowl = [tl.get_window_extent(rd) for tl in ax.get_yticklabels()]
        for i, (t, b) in enumerate(boxes):
            if b.x0 - 2 <= line_x <= b.x1 + 2: problems.append("%s: label '%s' touches the dashed line" % (tag, t.get_text()))
            for rl in rowl:
                if b.overlaps(rl): problems.append("%s: label '%s' overlaps a row label" % (tag, t.get_text()))
            for j, (t2, b2) in enumerate(boxes):
                if j > i and b.overlaps(b2): problems.append("%s: labels '%s' and '%s' overlap" % (tag, t.get_text(), t2.get_text()))
    def finish(fig, wrap, bottom_legend):
        fig.legend(handles=legend, loc="lower center", ncol=3 if bottom_legend else 1, frameon=False, fontsize=12.5, bbox_to_anchor=(0.5, 0.075 if bottom_legend else 0.06))
        fig.text(0.02, 0.0, textwrap.fill(FOOT, wrap), fontsize=11, va="top", style="italic", linespacing=1.5)
    figs = {}
    fig, axes = plt.subplots(1, 2, figsize=(22, 9.8)); fig.subplots_adjust(left=0.12, right=0.9, top=0.9, bottom=0.27, wspace=0.95)
    tb = draw(axes[0], "B"); tc = draw(axes[1], "C"); finish(fig, 215, True); check_overlaps(fig, axes[0], tb, "pair B"); check_overlaps(fig, axes[1], tc, "pair C"); figs["pair"] = fig
    fig2, ax2 = plt.subplots(figsize=(12.5, 11)); fig2.subplots_adjust(left=0.24, right=0.76, top=0.9, bottom=0.33)
    ta = draw(ax2, "A"); finish(fig2, 105, False); check_overlaps(fig2, ax2, ta, "A"); figs["A"] = fig2
    # stage flow v3
    a = ag[ag.r == 10].set_index("group"); N = int(a.n.sum()); mx = float(a.default_rate.max()); cm = plt.cm.Blues
    LAB = {"yes-yes": "Flagged at both stages", "yes-no": "Flagged at application only", "no-yes": "Flagged at monitoring only", "no-no": "Never flagged"}
    POS = {"yes-yes": (0, 1), "yes-no": (1, 1), "no-yes": (0, 0), "no-no": (1, 0)}
    fig3, ax3 = plt.subplots(figsize=(10, 7.8)); ax3.axis("off")
    for g, (cx, cy) in POS.items():
        r = a.loc[g]; dark = r.default_rate / mx > 0.55; tc_ = "white" if dark else "black"
        ax3.add_patch(Rectangle((cx + 0.02, cy + 0.02), 0.96, 0.96, fc=cm(0.15 + 0.8 * r.default_rate / mx), ec="white", lw=2))
        ax3.text(cx + 0.5, cy + 0.80, LAB[g], ha="center", va="center", fontsize=13, fontweight="bold", color=tc_)
        ax3.text(cx + 0.5, cy + 0.55, format(int(r.n), ","), ha="center", va="center", fontsize=30, color=tc_)
        ax3.text(cx + 0.5, cy + 0.33, "borrowers (%.1f%% of all)" % (100 * r.share_of_rows), ha="center", va="center", fontsize=11, color=tc_)
        ax3.text(cx + 0.5, cy + 0.15, "%.1f%% of them defaulted" % (100 * r.default_rate), ha="center", va="center", fontsize=12, fontweight="bold", color=tc_)
    ax3.text(0.5, 2.06, "Flagged at monitoring", ha="center", fontsize=12); ax3.text(1.5, 2.06, "Not flagged at monitoring", ha="center", fontsize=12)
    ax3.text(-0.04, 1.5, "Flagged at\napplication", ha="right", va="center", fontsize=12); ax3.text(-0.04, 0.5, "Not flagged at\napplication", ha="right", va="center", fontsize=12)
    ax3.set_ylim(0, 2.15); ax3.set_xlim(-0.4, 2)
    fig3.suptitle("Who is flagged at each stage? %s borrowers, flag threshold 1/11 (r = 10)" % format(N, ","), fontsize=13)
    fig3.text(0.02, 0.0, textwrap.fill("Darker tile = higher share of borrowers who defaulted. " + FOOT_FLOW, 130), fontsize=10, va="top", style="italic", linespacing=1.5); figs["flow"] = fig3
    if problems:
        print("COLLISIONS FOUND, nothing saved:"); [print("  ", p) for p in problems]; return
    print("collision check: no value label touches the dashed line, a row label, or another label (rendered extents)")
    out = {}
    for k, f in figs.items():
        b = io.BytesIO(); f.savefig(b, format="png", dpi=300, bbox_inches="tight"); out[k] = b.getvalue(); plt.close(f)
    # greyscale legibility check on the pair (not saved)
    im = Image.open(io.BytesIO(out["pair"])).convert("L"); arr = np.asarray(im).astype(float) / 255
    def lum(c):
        r, g, b = matplotlib.colors.to_rgb(c); return 0.299 * r + 0.587 * g + 0.114 * b
    print("greyscale check (pair): grey level of text/marker colours vs white background (0 = black, 1 = white); contrast = (1.05)/(L+0.05)")
    for nm, c in (("flip tag / diamond (vermilion)", FLIP), ("application blue", BLUE), ("monitoring green", GREEN), ("non-flip line grey", GREY), ("open-circle edge", "#707070"), ("value labels", "#333333")):
        L_ = lum(c); print("  %-34s grey %.2f | contrast %.1f:1 %s" % (nm, L_, 1.05 / (L_ + 0.05), "OK" if 1.05 / (L_ + 0.05) >= 3 else ("lines only (not text)" if nm.startswith("non-flip") else "LOW")))
    print("  rendered pair image: darkest pixel %.2f, share of non-white pixels %.3f (sanity: image is not blank)" % (arr.min(), float((arr < 0.97).mean())))
    print("  distinct marker shapes carry the flip signal (diamond vs circle) plus the text tag; colour is never the only signal")
    for k, b in out.items():
        p = ss.check_path(os.path.join(R, NM[k]))
        with open(p, "xb") as f: f.write(b)
        print("saved %-52s %7d bytes sha12 %s" % (NM[k], len(b), hashlib.sha256(b).hexdigest()[:12]))
    p = ss.check_path(os.path.join(R, NM["cap"]))
    with open(p, "x", encoding="utf-8") as f: f.write(CAP)
    print("saved", NM["cap"])
_c190()

In [ ]:
# Cell 191 (08_story_extras.ipynb): v7 with label-shift collision fix. Exclusive create. No restart. Replaces Cell 190 (nothing was saved by it).
def _c191():
    import os, sys, io, hashlib, textwrap, warnings
    import numpy as np, pandas as pd
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    from matplotlib.patches import Rectangle
    from matplotlib.ticker import FuncFormatter
    from PIL import Image
    warnings.filterwarnings("ignore")
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sys.path.insert(0, R + "/shared/src"); import safe_save as ss
    F = "selection/figures/"
    NM = {"pair": F + "sel_story_pair_BC_v7.png", "A": F + "sel_story_case_A_supplement_v7.png", "flow": F + "sel_stage_flow_v3.png", "cap": "selection/tables/sel_story_captions_v2.txt"}
    for n in NM.values(): ss.check_path(os.path.join(R, n))
    T = 1 / 11; BLUE, GREEN, FLIP, GREY = "#0072B2", "tab:green", "#D55E00", "#9a9a9a"; XMAX = 0.40
    ex = pd.read_csv(R + "/selection/tables/sel_worked_examples.csv"); ag = pd.read_csv(R + "/selection/tables/sel_stage_agreement.csv")
    NICE = {"Age": "Age", "Debt Ratio": "Debt ratio", "Credit Utilisation Rate": "Credit utilisation", "30-59 Days Past Due": "30–59 days past due", "60-89 Days Past Due": "60–89 days past due", "90+ Days Late": "90+ days past due"}
    TITLE = {"A": "Case A: flagged at both stages", "B": "Case B: not flagged at application, flagged at monitoring", "C": "Case C: flagged at both stages, close to the line"}
    FOOT = ("Sensitivity to unseen evidence, not the reason for a flag. Flag threshold 1/11 assumes a missed defaulter costs 10x a wrongly flagged borrower (r=10, assumed); at r=5 the application-stage model flags nobody. "
            "Cases chosen by rules fixed before outcomes were seen; illustrations, not typical borrowers. Fairness not assessed.")
    FOOT_FLOW = ("Flag threshold 1/11 assumes a missed defaulter costs 10x a wrongly flagged borrower (r=10, assumed); at r=5 the application-stage model flags nobody. "
                 "Application and monitoring flags come from two different models; flags are not decisions.")
    CAP = ("Each open marker removes ONE fact at a time. Rows are sorted by the shown shift |risk without the fact - risk with all facts| (the expected-shift S_exp order is in sel_worked_examples.csv).\n\n"
           "(i) Application and monitoring risks come from two different models (selection BN, 6 inputs; full BN, 10 inputs) answering different questions, so they need not agree.\n\n"
           "(ii) Facts not shown have exactly zero effect in this model (outside Default's Markov blanket in the learned structure).\n\n"
           "(iii) Overall mean predicted risk matches the observed rate (6.69% vs 6.68%); application-stage cell values are training default rates.\n\n"
           "(iv) 'Unknown' means the model averages over the possible values, weighted by how likely each is given the rest of the evidence, not set to zero.\n\n"
           "(v) Age is a direct input of the application-stage model; fairness not assessed; a flip shows the borrower is close to the line, not why they were flagged.\n\n"
           "Footer (printed on every worked-example figure): " + FOOT + "\n")
    pct = FuncFormatter(lambda v, _: "%d%%" % round(v * 100)); WB = dict(fc="white", ec="none", pad=1.5)
    problems = []
    def draw(ax, case):
        d = ex[ex.case == case].copy(); d["shown"] = (d.p_retracted - d.stage_baseline).abs(); ticks, labs, texts = [], [], []
        ax.axvspan(T, XMAX, color="#F3F3F3", zorder=0); ax.axvline(T, color="black", ls="--", lw=1.3, zorder=1)
        blocks = []
        for stage, lab, col in (("selection", "Application stage", BLUE), ("monitoring", "Monitoring stage", GREEN)):
            s = d[d.stage == stage].sort_values("shown", ascending=False); blocks.append((lab, col, float(s.stage_baseline.iloc[0]), s))
        n = sum(len(b[3]) + 2 for b in blocks); y = n
        ax.text(T - 0.006, n + 0.7, "not flagged", ha="right", fontsize=12, color="dimgray"); ax.text(T + 0.006, n + 0.7, "flagged", ha="left", fontsize=12, color="dimgray")
        for lab, col, base, s in blocks:
            y -= 0.6; ax.text(0.005, y + 0.62, "%s: risk %.1f%% (%s)" % (lab, 100 * base, "flagged" if base >= T else "not flagged"), fontsize=14, fontweight="bold", va="center", ha="left", color=col); y -= 1.3
            for r in s.itertuples():
                flip = bool(r.realised_flip); c = FLIP if flip else GREY; ticks.append(y); labs.append((NICE[r.variable], flip))
                ax.plot([base, r.p_retracted], [y, y], color=c, lw=4.0 if flip else 1.8, zorder=2, solid_capstyle="butt")
                ax.scatter([r.p_retracted], [y], s=250 if flip else 210, marker="D" if flip else "o", facecolor="white", edgecolor=FLIP if flip else "#707070", linewidth=2.8, zorder=4)
                ax.scatter([base], [y], s=150, color=col, zorder=5, edgecolor="white", linewidth=1.2)
                t = ax.text(r.p_retracted, y + 0.42, ("no visible change" if r.shown < 0.02 else "%.1f%%" % (100 * r.p_retracted)), ha="center", va="bottom", fontsize=12, color="#333333", zorder=6, bbox=WB)
                texts.append((t, r.p_retracted))
                if flip: ax.text(XMAX + 0.012, y, "FLAG FLIPS IF UNKNOWN", va="center", ha="left", fontsize=12.5, fontweight="bold", color=FLIP, clip_on=False)
                y -= 1
        ax.set_yticks(ticks); ax.set_yticklabels([l for l, _ in labs], fontsize=14)
        for tl, (_, fl) in zip(ax.get_yticklabels(), labs):
            if fl: tl.set_color(FLIP); tl.set_fontweight("bold")
        ax.set_xlim(0, XMAX); ax.set_ylim(y + 0.2, n + 1.4); ax.xaxis.set_major_formatter(pct); ax.tick_params(axis="x", labelsize=13); ax.tick_params(axis="y", length=0, pad=8)
        ax.set_xlabel("Estimated probability of default", fontsize=14); ax.grid(False)
        for sp in ("top", "right", "left"): ax.spines[sp].set_visible(False)
        ax.set_title(TITLE[case], fontsize=16, pad=16, loc="left"); return texts
    legend = [Line2D([0], [0], marker="o", color="w", markerfacecolor=BLUE, markersize=12, label="blue / green dot: risk with all facts known (application / monitoring)"),
              Line2D([0], [0], marker="o", color="w", markerfacecolor="white", markeredgecolor="#707070", markeredgewidth=2.5, markersize=12, label="open circle: risk if that one fact were unknown"),
              Line2D([0], [0], marker="D", color=FLIP, markerfacecolor="white", markeredgewidth=2.5, markersize=11, lw=3, label="open diamond + orange line + tag: the flag would flip")]
    def fix_and_check(fig, ax, texts, tag):
        fig.canvas.draw(); rd = fig.canvas.get_renderer(); line_x = ax.transData.transform((T, 0))[0]; inv = ax.transData.inverted()
        rowl = [tl.get_window_extent(rd) for tl in ax.get_yticklabels()]
        for t, px in texts:
            b = t.get_window_extent(rd); gap = 10
            if b.x0 - gap <= line_x <= b.x1 + gap:
                w = b.x1 - b.x0; cx = ax.transData.transform((px, 0))[0]
                t.set_ha("right" if cx <= line_x else "left")
                t.set_position((inv.transform((line_x - gap if cx <= line_x else line_x + gap, 0))[0], t.get_position()[1])) if False else None
                # shift so the box clears the line: right-aligned ends gap px left of the line, left-aligned starts gap px right of it
                xtarget = inv.transform((line_x - gap, 0))[0] if cx <= line_x else inv.transform((line_x + gap, 0))[0]
                t.set_position((xtarget, t.get_position()[1]))
        fig.canvas.draw(); rd = fig.canvas.get_renderer(); boxes = [(t, t.get_window_extent(rd)) for t, _ in texts]
        for i, (t, b) in enumerate(boxes):
            if b.x0 <= line_x <= b.x1: problems.append("%s: label '%s' still touches the dashed line" % (tag, t.get_text()))
            for rl in rowl:
                if b.overlaps(rl): problems.append("%s: label '%s' overlaps a row label" % (tag, t.get_text()))
            for j, (t2, b2) in enumerate(boxes):
                if j > i and b.overlaps(b2): problems.append("%s: labels '%s' and '%s' overlap" % (tag, t.get_text(), t2.get_text()))
    def finish(fig, wrap, bottom_legend):
        fig.legend(handles=legend, loc="lower center", ncol=3 if bottom_legend else 1, frameon=False, fontsize=12.5, bbox_to_anchor=(0.5, 0.075 if bottom_legend else 0.06))
        fig.text(0.02, 0.0, textwrap.fill(FOOT, wrap), fontsize=11, va="top", style="italic", linespacing=1.5)
    figs = {}
    fig, axes = plt.subplots(1, 2, figsize=(22, 9.8)); fig.subplots_adjust(left=0.12, right=0.9, top=0.9, bottom=0.27, wspace=0.95)
    tb = draw(axes[0], "B"); tc = draw(axes[1], "C"); finish(fig, 215, True); fix_and_check(fig, axes[0], tb, "pair B"); fix_and_check(fig, axes[1], tc, "pair C"); figs["pair"] = fig
    fig2, ax2 = plt.subplots(figsize=(12.5, 11)); fig2.subplots_adjust(left=0.24, right=0.76, top=0.9, bottom=0.33)
    ta = draw(ax2, "A"); finish(fig2, 105, False); fix_and_check(fig2, ax2, ta, "A"); figs["A"] = fig2
    a = ag[ag.r == 10].set_index("group"); N = int(a.n.sum()); mx = float(a.default_rate.max()); cm = plt.cm.Blues
    LAB = {"yes-yes": "Flagged at both stages", "yes-no": "Flagged at application only", "no-yes": "Flagged at monitoring only", "no-no": "Never flagged"}
    POS = {"yes-yes": (0, 1), "yes-no": (1, 1), "no-yes": (0, 0), "no-no": (1, 0)}
    fig3, ax3 = plt.subplots(figsize=(10, 7.8)); ax3.axis("off")
    for g, (cx, cy) in POS.items():
        r = a.loc[g]; tc_ = "white" if r.default_rate / mx > 0.55 else "black"
        ax3.add_patch(Rectangle((cx + 0.02, cy + 0.02), 0.96, 0.96, fc=cm(0.15 + 0.8 * r.default_rate / mx), ec="white", lw=2))
        ax3.text(cx + 0.5, cy + 0.80, LAB[g], ha="center", va="center", fontsize=13, fontweight="bold", color=tc_)
        ax3.text(cx + 0.5, cy + 0.55, format(int(r.n), ","), ha="center", va="center", fontsize=30, color=tc_)
        ax3.text(cx + 0.5, cy + 0.33, "borrowers (%.1f%% of all)" % (100 * r.share_of_rows), ha="center", va="center", fontsize=11, color=tc_)
        ax3.text(cx + 0.5, cy + 0.15, "%.1f%% of them defaulted" % (100 * r.default_rate), ha="center", va="center", fontsize=12, fontweight="bold", color=tc_)
    ax3.text(0.5, 2.06, "Flagged at monitoring", ha="center", fontsize=12); ax3.text(1.5, 2.06, "Not flagged at monitoring", ha="center", fontsize=12)
    ax3.text(-0.04, 1.5, "Flagged at\napplication", ha="right", va="center", fontsize=12); ax3.text(-0.04, 0.5, "Not flagged at\napplication", ha="right", va="center", fontsize=12)
    ax3.set_ylim(0, 2.15); ax3.set_xlim(-0.4, 2)
    fig3.suptitle("Who is flagged at each stage? %s borrowers, flag threshold 1/11 (r = 10)" % format(N, ","), fontsize=13)
    fig3.text(0.02, 0.0, textwrap.fill("Darker tile = higher share of borrowers who defaulted. " + FOOT_FLOW, 130), fontsize=10, va="top", style="italic", linespacing=1.5); figs["flow"] = fig3
    if problems:
        print("COLLISIONS FOUND, nothing saved:"); [print("  ", p) for p in problems]; return
    print("collision check: after label shifting, no value label touches the dashed line, a row label, or another label (rendered extents)")
    out = {}
    for k, f in figs.items():
        b = io.BytesIO(); f.savefig(b, format="png", dpi=300, bbox_inches="tight"); out[k] = b.getvalue(); plt.close(f)
    def lum(c):
        r, g, b = matplotlib.colors.to_rgb(c); return 0.299 * r + 0.587 * g + 0.114 * b
    print("greyscale check (pair; not saved): contrast of each colour vs white = 1.05/(L+0.05)")
    for nm, c in (("flip tag / diamond (vermilion)", FLIP), ("application blue", BLUE), ("monitoring green", GREEN), ("non-flip line grey", GREY), ("open-circle edge", "#707070"), ("value labels", "#333333")):
        L_ = lum(c); ct = 1.05 / (L_ + 0.05); print("  %-34s grey %.2f | contrast %.1f:1 %s" % (nm, L_, ct, "OK" if ct >= 3 else ("lines only, not text" if nm.startswith("non-flip") else "LOW")))
    arr = np.asarray(Image.open(io.BytesIO(out["pair"])).convert("L")).astype(float) / 255
    print("  rendered pair: darkest pixel %.2f, non-white share %.3f; flip signal = diamond marker + text tag + orange line, so colour is never the only signal" % (arr.min(), float((arr < 0.97).mean())))
    for k, b in out.items():
        p = ss.check_path(os.path.join(R, NM[k]))
        with open(p, "xb") as f: f.write(b)
        print("saved %-52s %7d bytes sha12 %s" % (NM[k], len(b), hashlib.sha256(b).hexdigest()[:12]))
    p = ss.check_path(os.path.join(R, NM["cap"]))
    with open(p, "x", encoding="utf-8") as f: f.write(CAP)
    print("saved", NM["cap"])
_c191()

In [ ]:
# Cell 192 (08_story_extras.ipynb): DRY RUN for moving superseded figures into selection/figures/_superseded/. Writes NOTHING.
def _c192():
    import os, csv
    R = os.environ["CREDIT_BN_ROOT"]
    F = R + "/selection/figures/"
    MOVE = ["sel_worked_fan.png", "sel_worked_journey.png", "sel_worked_age_bubbles.png", "sel_stage_flow.png", "sel_stage_flow_v2.png",
            "sel_story_case_A_v2.png", "sel_story_case_B_v2.png", "sel_story_case_C_v2.png",
            "sel_story_pair_BC_v4.png", "sel_story_pair_BC_v5.png", "sel_story_pair_BC_v6.png",
            "sel_story_case_A_supplement_v4.png", "sel_story_case_A_supplement_v5.png", "sel_story_case_A_supplement_v6.png"]
    KEEP = ["sel_story_case_A_v3.png", "sel_story_case_B_v3.png", "sel_story_case_C_v3.png", "sel_story_pair_BC_v7.png", "sel_story_case_A_supplement_v7.png", "sel_stage_flow_v3.png", "sel_decision_value_combined.png"]
    print("DRY_RUN = True")
    print("destination folder exists:", os.path.isdir(F + "_superseded"))
    for f in MOVE: print("  %-8s %s" % ("MOVE" if os.path.exists(F + f) else "MISSING", f))
    for f in KEEP: print("  %-8s %s" % ("STAYS" if os.path.exists(F + f) else "MISSING", f))
    clash = [f for f in MOVE if os.path.exists(F + "_superseded/" + f)]
    print("name clashes in destination:", clash or "none")
    print("other PNGs in selection/figures NOT in either list:", sorted(f for f in os.listdir(F) if f.endswith(".png") and f not in MOVE + KEEP))
    print("DRY RUN ONLY: nothing moved.")
_c192()

In [ ]:
# Cell 193 (08_story_extras.ipynb): REAL MOVE of superseded figures into selection/figures/_superseded/. No deletes, no overwrites. Appends manifest rows + log. No restart.
def _c193():
    import os, csv, time, hashlib, subprocess
    R = os.environ["CREDIT_BN_ROOT"]
    F = R + "/selection/figures/"; D = F + "_superseded/"
    MOVE = ["sel_worked_fan.png", "sel_worked_journey.png", "sel_worked_age_bubbles.png", "sel_stage_flow.png", "sel_stage_flow_v2.png",
            "sel_story_case_A_v2.png", "sel_story_case_B_v2.png", "sel_story_case_C_v2.png",
            "sel_story_pair_BC_v4.png", "sel_story_pair_BC_v5.png", "sel_story_pair_BC_v6.png",
            "sel_story_case_A_supplement_v4.png", "sel_story_case_A_supplement_v5.png", "sel_story_case_A_supplement_v6.png",
            "sel_dag_9node.png", "sel_dag_9node_v2.png"]
    git = lambda *a: subprocess.run(["git", "-C", R, *a], capture_output=True, text=True).stdout.strip()
    sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
    mp = R + "/MANIFEST.csv"; raw0 = open(mp, "rb").read(); recs = list(csv.DictReader(open(mp, newline="")))
    assert raw0.endswith(b"\n") and len(recs) == 238, "unexpected manifest state: STOP"
    assert git("rev-list", "-n", "1", "full-variable-final").startswith("5626192403ef"), "tag moved: STOP"
    for f in MOVE:
        assert os.path.exists(F + f), "missing %s: STOP" % f
    os.makedirs(D, exist_ok=False)
    for f in MOVE:
        assert not os.path.exists(D + f)
    before = {f: sha(F + f) for f in MOVE}; log = []; rows = []; now = time.strftime("%Y-%m-%d %H:%M")
    for f in MOVE:
        os.rename(F + f, D + f)
        assert sha(D + f) == before[f] and not os.path.exists(F + f)
        log.append([f, "selection/figures/" + f, "selection/figures/_superseded/" + f, before[f][:12], now])
        rows.append(["selection/figures/_superseded/" + f, os.path.getsize(D + f), before[f], "selection", now, "MOVED from selection/figures/%s on 2026-10-17; SUPERSEDED, kept" % f])
    lp = R + "/selection/tables/sel_superseded_figures_log.csv"
    with open(lp, "x", newline="") as fh:
        w = csv.writer(fh); w.writerow(["file", "old_path", "new_path", "sha12", "moved_at"]); w.writerows(log)
    with open(mp, "a", newline="") as fh: csv.writer(fh).writerows(rows)
    assert len(list(csv.DictReader(open(mp, newline="")))) == 238 + len(rows) and open(mp, "rb").read().startswith(raw0)
    print("moved %d files (hashes verified after move); log: selection/tables/sel_superseded_figures_log.csv; manifest rows now %d" % (len(MOVE), 238 + len(rows)))
    print("figures still in selection/figures:", sorted(f for f in os.listdir(F) if f.endswith(".png")))
_c193()

In [ ]:
# Cell 194 (08_story_extras.ipynb): manifest/git DRY RUN for the v7 + move batch. Writes NOTHING.
def _c194():
    import os, csv, hashlib, subprocess
    R = os.environ["CREDIT_BN_ROOT"]
    git = lambda *a: subprocess.run(["git", "-C", R, *a], capture_output=True, text=True).stdout.strip()
    sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
    mp = R + "/MANIFEST.csv"; recs = list(csv.DictReader(open(mp, newline="")))
    have = {r["path"] for r in recs}
    print("DRY_RUN = True")
    print("manifest rows now:", len(recs), "(expected 254) | HEAD rows:", len(list(csv.DictReader(git("show", "HEAD:MANIFEST.csv").splitlines()))))
    print("tag OK:", git("rev-list", "-n", "1", "full-variable-final").startswith("5626192403ef"))
    print("git status:", git("status", "--porcelain").splitlines())
    new = []
    for top in ("selection", "shared", "monitoring_rq3"):
        for dp, dn, fn in os.walk(R + "/" + top):
            dn[:] = [d for d in dn if d not in (".ipynb_checkpoints", "__pycache__")]
            for f in sorted(fn):
                rel = os.path.relpath(os.path.join(dp, f), R)
                if rel not in have and not f.endswith(".pyc") and f != ".DS_Store": new.append((rel, os.path.getsize(os.path.join(dp, f)), sha(os.path.join(dp, f))[:12]))
    print("\nfiles on disk not yet in the manifest (%d):" % len(new)); [print("  ", *x) for x in sorted(new)]
    print("\nwould stage ONLY: MANIFEST.csv and selection/notebooks/08_story_extras.ipynb")
    print("DRY RUN ONLY: nothing written.")
_c194()

In [ ]:
# Cell 195 (08_story_extras.ipynb): REAL RUN. Manifest append (5 rows) + NEW commit (MANIFEST.csv + 08 notebook only). Tag never touched.
def _c195():
    import os, csv, time, hashlib, subprocess
    R = os.environ["CREDIT_BN_ROOT"]
    NB = "selection/notebooks/08_story_extras.ipynb"; now = time.strftime("%Y-%m-%d %H:%M"); NOTE = "added after Phase 7b figure pass, 2026-10-17"
    def git(*a, check=True):
        r = subprocess.run(["git", "-C", R, *a], capture_output=True, text=True)
        if check and r.returncode != 0: raise RuntimeError("git %s failed: %s" % (" ".join(a), r.stderr.strip()[:300]))
        return r.stdout.strip()
    sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
    mp = R + "/MANIFEST.csv"; raw0 = open(mp, "rb").read(); recs = list(csv.DictReader(open(mp, newline="")))
    assert raw0.endswith(b"\n") and len(recs) == 254, "unexpected manifest state: STOP"
    assert git("rev-list", "-n", "1", "full-variable-final").startswith("5626192403ef"), "tag moved: STOP"
    F = [("selection/figures/sel_stage_flow_v3.png", "be2baaf2e89c", "; stage flow with footer (r=10 assumed; flags are not decisions)"),
         ("selection/figures/sel_story_case_A_supplement_v7.png", "3ba72e4f1b60", "; v7 supplement (v3 layout, new wording/footer). Known: greyscale contrast of orange tag and blue/green headers below 3:1; flip also carried by diamond marker + text tag"),
         ("selection/figures/sel_story_pair_BC_v7.png", "54c9e73b90de", "; v7 poster pair B+C (v3 layout, new wording/footer). Same greyscale note"),
         ("selection/tables/sel_story_captions_v2.txt", "b0250ce5ac8e", "; caption facts (i)-(v); replaces sel_story_captions.txt (kept)"),
         ("selection/tables/sel_superseded_figures_log.csv", "57646ba17397", "; log of 16 superseded figures moved to selection/figures/_superseded/")]
    have = {r["path"] for r in recs}; rows = []
    for rel, h12, extra in F:
        p = R + "/" + rel; assert rel not in have and os.path.exists(p), "unexpected state for %s: STOP" % rel
        h = sha(p); assert h.startswith(h12), "hash changed for %s: STOP" % rel
        rows.append([rel, os.path.getsize(p), h, "selection", now, NOTE + extra])
    with open(mp, "a", newline="") as fh: csv.writer(fh).writerows(rows)
    n = len(list(csv.DictReader(open(mp, newline=""))))
    assert n == 259 and open(mp, "rb").read().startswith(raw0), "manifest check failed: STOP (not committed)"
    git("add", "MANIFEST.csv", NB)
    ident = [] if (git("config", "user.name", check=False) and git("config", "user.email", check=False)) else ["-c", "user.name=Ahmed Nanabhay", "-c", "user.email=noreply@local.invalid"]
    git(*ident, "commit", "-q", "-m", "Phase 7b: v7 story figures, stage flow v3, captions v2, superseded figures moved and logged")
    print("manifest rows now:", n, "(expected 259)"); print("new commit:", git("log", "--oneline", "-1"))
    print("tag unchanged:", git("rev-list", "-n", "1", "full-variable-final").startswith("5626192403ef")); print("git status:", git("status", "--porcelain").splitlines())
_c195()